# 实验 008：因果补全后的全量特征与排序模型

这个 Notebook 按照 `读取 → 因果补全全部 X → 特征缓存 → 时间策略验证 → 手动模型选择 → 最终训练` 的顺序执行。`mask_x` 只标记原始数据来源；补全完成后，正式特征与横截面排名统一使用 `filled_num_x`，样本范围由 `mask_y` 决定。

> 注意：`data.z` 体积较大，完整读取和解压会占用较多内存，请确认机器内存充足后再运行读取单元格。

## 1. 导入常用库

In [1]:
import gc
import hashlib
import json
import pickle
import random
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zstandard as zstd
from scipy.stats import rankdata, spearmanr

EXPECTED_CONDA_ENV = "jingge_ts"
CURRENT_CONDA_ENV = Path(sys.prefix).name
if CURRENT_CONDA_ENV.lower() != EXPECTED_CONDA_ENV.lower():
    raise RuntimeError(
        f"请把 Notebook 内核切换到 Anaconda 环境 {EXPECTED_CONDA_ENV}；"
        f"当前解释器为 {sys.executable}"
    )
print(f"Anaconda 内核：{sys.executable}")
print(f"LightGBM：{lgb.__version__}")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

Anaconda 内核：d:\anaconda\anaconda_data\envs\jingge_ts\python.exe
LightGBM：4.7.0


## 2. 配置输入与输出路径

无论从项目根目录还是当前实验目录启动 Notebook，下面的代码都会自动定位项目根目录。后续如需修改文件名，只需要调整这个单元格。

In [2]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data.z").exists() and (candidate / "02_experiments").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录，请从项目目录或实验目录启动 Notebook。")


PROJECT_ROOT = find_project_root()
EXPERIMENT_ID = "exp_008_new_method"

# 输入
DATA_PATH = PROJECT_ROOT / "data.z"

# 当前实验及可选缓存
EXPERIMENT_DIR = PROJECT_ROOT / "02_experiments" / EXPERIMENT_ID
CACHE_DIR = PROJECT_ROOT / "03_cache" / EXPERIMENT_ID

# 统一输出
OUTPUT_DIR = PROJECT_ROOT / "04_results" / EXPERIMENT_ID
PREDICTION_PATH = OUTPUT_DIR / "prediction.npy"
METRICS_PATH = OUTPUT_DIR / "metrics.json"
MODEL_PATH = OUTPUT_DIR / "model.txt"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

path_summary = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "data_path": str(DATA_PATH),
        "experiment_dir": str(EXPERIMENT_DIR),
        "cache_dir": str(CACHE_DIR),
        "output_dir": str(OUTPUT_DIR),
        "prediction_path": str(PREDICTION_PATH),
        "metrics_path": str(METRICS_PATH),
        "model_path": str(MODEL_PATH),
    },
    name="paths",
)
path_summary

project_root                                   D:\google_dl\book\友安杯
data_path                               D:\google_dl\book\友安杯\data.z
experiment_dir     D:\google_dl\book\友安杯\02_experiments\exp_008_n...
cache_dir          D:\google_dl\book\友安杯\03_cache\exp_008_new_method
output_dir         D:\google_dl\book\友安杯\04_results\exp_008_new_m...
prediction_path    D:\google_dl\book\友安杯\04_results\exp_008_new_m...
metrics_path       D:\google_dl\book\友安杯\04_results\exp_008_new_m...
model_path         D:\google_dl\book\友安杯\04_results\exp_008_new_m...
Name: paths, dtype: object

## 3. 按官方格式读取 `data.z`

`data.z` 保存的是经过 Zstandard 压缩的 Pickle 数据。这里沿用项目文档中的常规读取方式：先读取压缩字节，再解压并反序列化。

In [3]:
def load_data_z(path: Path) -> dict:
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"找不到输入文件：{path}")

    started_at = time.time()
    print(f"正在读取：{path}")

    compressed_bytes = pd.read_pickle(path)
    if not isinstance(compressed_bytes, (bytes, bytearray)):
        raise TypeError("data.z 的外层对象不是 bytes，文件格式可能与文档不一致。")

    decompressed_bytes = zstd.ZstdDecompressor().decompress(compressed_bytes)
    del compressed_bytes
    gc.collect()

    loaded_data = pickle.loads(decompressed_bytes)
    del decompressed_bytes
    gc.collect()

    if not isinstance(loaded_data, dict):
        raise TypeError("解压后的数据不是字典，请检查输入文件。")

    print(f"读取完成，用时 {(time.time() - started_at) / 60:.2f} 分钟。")
    return loaded_data


data = load_data_z(DATA_PATH)

正在读取：D:\google_dl\book\友安杯\data.z
读取完成，用时 0.22 分钟。


## 4. 检查数据结构与官方切分

In [4]:
structure_rows = []
for key, value in data.items():
    structure_rows.append(
        {
            "key": key,
            "python_type": type(value).__name__,
            "shape": getattr(value, "shape", None),
            "dtype": str(getattr(value, "dtype", "")),
        }
    )

structure_df = pd.DataFrame(structure_rows)
structure_df

,key,python_type,shape,dtype
0,num_x,ndarray,"(3603, 5282, 99)",float32
1,cat_x,ndarray,"(3603, 5282, 9)",int64
2,y1,ndarray,"(3603, 5282)",float32
3,y2,ndarray,"(3603, 5282)",float32
4,mask_x,ndarray,"(3603, 5282)",bool
5,mask_y,ndarray,"(3603, 5282)",bool
6,train_start_idx,int64,(),int64
7,valid_start_idx,int64,(),int64
8,test_start_idx,int64,(),int64


In [5]:
T, STOCK_COUNT, NUMERIC_FEATURE_COUNT = data["num_x"].shape
CATEGORY_FEATURE_COUNT = data["cat_x"].shape[2]

TRAIN_START = int(data["train_start_idx"])
VALID_START = int(data["valid_start_idx"])
TEST_START = int(data["test_start_idx"])

assert data["cat_x"].shape[:2] == (T, STOCK_COUNT)
assert data["y1"].shape == (T, STOCK_COUNT)
assert data["mask_x"].shape == (T, STOCK_COUNT)
assert data["mask_y"].shape == (T, STOCK_COUNT)
assert 0 <= TRAIN_START <= VALID_START <= TEST_START <= T

SPLITS = {
    "pretrain": slice(0, TRAIN_START),
    "train": slice(TRAIN_START, VALID_START),
    "valid": slice(VALID_START, TEST_START),
    "test": slice(TEST_START, T),
}

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "start": split_slice.start,
            "stop": split_slice.stop,
            "time_points": split_slice.stop - split_slice.start,
        }
        for name, split_slice in SPLITS.items()
    ]
)

print(
    f"T={T}, stocks={STOCK_COUNT}, "
    f"numeric_features={NUMERIC_FEATURE_COUNT}, "
    f"categorical_features={CATEGORY_FEATURE_COUNT}"
)
split_summary

T=3603, stocks=5282, numeric_features=99, categorical_features=9


,split,start,stop,time_points
0,pretrain,0,486,486
1,train,486,2918,2432
2,valid,2918,3161,243
3,test,3161,3603,442


## 5. 常用数据引用与输出辅助函数

训练或验证时，样本范围由 `mask_y & np.isfinite(y1)` 决定。`mask_x` 仅用于下一部分识别需要补全的原始 X；正式特征不再用它筛选样本。

In [6]:
num_x = data["num_x"]
cat_x = data["cat_x"]
y1 = data["y1"]
mask_x = data["mask_x"]
mask_y = data["mask_y"]


def supervised_mask(split_name: str) -> np.ndarray:
    split_slice = SPLITS[split_name]
    return mask_y[split_slice] & np.isfinite(y1[split_slice])


def save_prediction(prediction: np.ndarray, path: Path = PREDICTION_PATH) -> Path:
    prediction = np.asarray(prediction, dtype=np.float32)
    expected_shape = (T - TEST_START, STOCK_COUNT)
    if prediction.shape != expected_shape:
        raise ValueError(f"预测形状应为 {expected_shape}，实际为 {prediction.shape}")
    if not np.all(np.isfinite(prediction)):
        raise ValueError("预测结果包含 NaN 或无穷值。")

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial_path = path.with_suffix(path.suffix + ".partial")
    with partial_path.open("wb") as handle:
        np.save(handle, prediction)
    partial_path.replace(path)
    print(f"预测已保存：{path.resolve()}")
    return path


def save_metrics(metrics: dict, path: Path = METRICS_PATH) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(f"指标已保存：{path.resolve()}")
    return path

## 6. 第一部分：样本资格与历史连续性

本节落实已经确认的样本处理原则：

1. 原始 `mask_x`、`mask_y` 永不修改；`mask_y` 决定样本，`mask_x` 只记录真实或补全来源。
2. 首次真实观测之前不做趋势外推；只补首次观测之后、长度不超过 30 的内部短缺口。
3. 补全只使用当前及过去信息，并保留真实/补全/不可用状态、缺口长度、可信度和历史覆盖率。
4. 默认使用每个时间点的全部有效股票；只有内存不足时才启用可复现的分层抽样。
5. 近期历史默认保留 30 个时间点，长期历史只预留稀疏位置，具体特征将在后续部分讨论。

In [7]:
@dataclass(frozen=True)
class GapImputationConfig:
    # 只把长度不超过 30 的内部缺口视为可补全历史。
    max_gap: int = 30
    # 使用最近若干个真实观测间的单位时间变化估计趋势。
    drift_observations: int = 10
    drift_ewma_alpha: float = 0.35
    # 线性趋势最多外推 5 步，之后主要依靠可信度衰减。
    trend_step_cap: int = 5
    confidence_tau: float = 10.0
    # 长缺口不删除股票，而是逐渐回归当期市场中位数。
    long_gap_tau: float = 30.0
    # 极值保护参数只由 Train 拟合。
    lower_quantile: float = 0.005
    upper_quantile: float = 0.995
    guardrail_time_samples: int = 256
    guardrail_stocks_per_time: int = 512

    def __post_init__(self) -> None:
        if self.max_gap <= 0 or self.trend_step_cap <= 0:
            raise ValueError("缺口长度与趋势步数必须为正数。")
        if not (0.0 < self.drift_ewma_alpha <= 1.0):
            raise ValueError("drift_ewma_alpha 必须位于 (0, 1]。")
        if self.confidence_tau <= 0 or self.long_gap_tau <= 0:
            raise ValueError("补全可信度衰减参数必须为正数。")


GAP_CONFIG = GapImputationConfig()

# None 表示正式训练默认使用每个时间点的全部有效股票。
TRAIN_STOCK_CAP = None
TRAIN_QUERY_STRATA = 20

# 输入窗口约定：[t-29, t] 恰好包含 30 个时间点。
DENSE_HISTORY_LENGTH = 30
LONG_HISTORY_LAGS = (60, 120, 240)

SAMPLE_GUARDRAIL_STOP = TRAIN_START + int(round((VALID_START - TRAIN_START) * 0.60))
MARKET_CENTER_PATH = CACHE_DIR / "cross_sectional_median.npy"
FILLED_NUMERIC_CACHE_ROOT = CACHE_DIR / "filled_numeric_v2"
REBUILD_FILLED_NUMERIC_CACHE = False
RUN_ARTIFICIAL_GAP_CHECK = False

pd.Series(
    {
        "max_gap": GAP_CONFIG.max_gap,
        "dense_history_length": DENSE_HISTORY_LENGTH,
        "long_history_lags": LONG_HISTORY_LAGS,
        "train_stock_cap": TRAIN_STOCK_CAP,
        "guardrail_fit_interval": (TRAIN_START, SAMPLE_GUARDRAIL_STOP),
        "market_center_path": str(MARKET_CENTER_PATH),
    },
    name="sample_processing_config",
)

max_gap                                                                  30
dense_history_length                                                     30
long_history_lags                                            (60, 120, 240)
train_stock_cap                                                        None
guardrail_fit_interval                                          (486, 1945)
market_center_path        D:\google_dl\book\友安杯\03_cache\exp_008_new_met...
Name: sample_processing_config, dtype: object

### 6.1 审计掩码与股票生命周期

补全之前先审计数据来源。`mask_y=True` 的位置进入相应训练、验证或预测股票池；其中 `mask_x=False` 的 X 会先完成因果补全，再与真实 X 一样进入特征和横截面排名。

In [8]:
mask_audit_rows = []
for split_name, split_slice in SPLITS.items():
    split_mask_x = mask_x[split_slice]
    split_mask_y = mask_y[split_slice]
    split_y1 = y1[split_slice]
    supervised = split_mask_y & np.isfinite(split_y1)
    mask_audit_rows.append(
        {
            "split": split_name,
            "positions": int(split_mask_x.size),
            "mask_x_true": int(np.count_nonzero(split_mask_x)),
            "mask_y_true": int(np.count_nonzero(split_mask_y)),
            "supervised_rows": int(np.count_nonzero(supervised)),
            "rows_requiring_imputation": int(
                np.count_nonzero(split_mask_y & ~split_mask_x)
            ),
        }
    )

mask_audit_df = pd.DataFrame(mask_audit_rows)

has_observation = mask_x.any(axis=0)
first_valid_time = np.full(STOCK_COUNT, -1, dtype=np.int32)
last_valid_time = np.full(STOCK_COUNT, -1, dtype=np.int32)
first_valid_time[has_observation] = np.argmax(
    mask_x[:, has_observation], axis=0
).astype(np.int32)
last_valid_time[has_observation] = (
    T - 1 - np.argmax(mask_x[::-1, has_observation], axis=0)
).astype(np.int32)

print("mask_y 决定样本范围；mask_x=False 的 X 将在下一步全部补全。")
mask_audit_df

mask_y 决定样本范围；mask_x=False 的 X 将在下一步全部补全。


,split,positions,mask_x_true,mask_y_true,supervised_rows,rows_requiring_imputation
0,pretrain,2567052,866368,723781,0,1
1,train,12845824,7354184,6489099,6489099,0
2,valid,1283526,1089602,982972,982972,0
3,test,2334644,2219158,2042538,0,0


### 6.2 拟合 Train 极值保护与横截面中性水平

极值上下界只从 Train 的确定性分层样本拟合。横截面中位数只使用同一时间点其他真实有效股票，可以按需计算，也可以一次建立小型缓存供后续重复使用。

In [9]:
def fit_train_guardrails(
    numeric_values: np.ndarray,
    observed_mask: np.ndarray,
    train_start: int,
    train_stop: int,
    config: GapImputationConfig = GAP_CONFIG,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray, int]:
    time_count = min(config.guardrail_time_samples, train_stop - train_start)
    sampled_times = np.linspace(
        train_start, train_stop - 1, time_count, dtype=np.int64
    )
    sampled_rows = []

    for time_idx in sampled_times:
        eligible = np.flatnonzero(observed_mask[time_idx])
        if eligible.size > config.guardrail_stocks_per_time:
            rng = np.random.default_rng(seed + int(time_idx) * 1009)
            eligible = np.sort(
                rng.choice(
                    eligible,
                    size=config.guardrail_stocks_per_time,
                    replace=False,
                )
            )
        if eligible.size:
            sampled_rows.append(
                np.asarray(numeric_values[time_idx, eligible], dtype=np.float32)
            )

    if not sampled_rows:
        raise RuntimeError("Train 中没有可用于拟合极值保护的真实样本。")

    sample = np.concatenate(sampled_rows, axis=0)
    lower, upper = np.quantile(
        sample,
        [config.lower_quantile, config.upper_quantile],
        axis=0,
    ).astype(np.float32)
    if not np.all(np.isfinite(lower)) or not np.all(np.isfinite(upper)):
        raise ValueError("极值保护包含非有限值。")
    return lower, upper, int(sample.shape[0])


def compute_cross_sectional_median(
    numeric_values: np.ndarray,
    observed_mask: np.ndarray,
    start: int = 0,
    stop: int | None = None,
) -> np.ndarray:
    stop = numeric_values.shape[0] if stop is None else int(stop)
    feature_count = numeric_values.shape[2]
    centers = np.zeros((stop - start, feature_count), dtype=np.float32)

    for output_idx, time_idx in enumerate(range(start, stop)):
        eligible = observed_mask[time_idx]
        if np.any(eligible):
            centers[output_idx] = np.median(
                numeric_values[time_idx, eligible], axis=0
            ).astype(np.float32)
        elif output_idx > 0:
            centers[output_idx] = centers[output_idx - 1]
    return centers


def load_or_build_market_center(
    path: Path = MARKET_CENTER_PATH,
) -> np.ndarray:
    path = Path(path)
    if path.exists():
        cached = np.load(path, mmap_mode="r")
        if cached.shape != (T, NUMERIC_FEATURE_COUNT):
            raise ValueError(f"市场中位数缓存形状错误：{cached.shape}")
        return cached

    centers = compute_cross_sectional_median(num_x, mask_x, 0, T)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial_path = path.with_suffix(path.suffix + ".partial")
    with partial_path.open("wb") as handle:
        np.save(handle, centers)
    partial_path.replace(path)
    return np.load(path, mmap_mode="r")


TRAIN_LOWER, TRAIN_UPPER, GUARDRAIL_SAMPLE_ROWS = fit_train_guardrails(
    num_x, mask_x, TRAIN_START, SAMPLE_GUARDRAIL_STOP
)
print(
    f"早期 Train 极值保护拟合完成：区间 [{TRAIN_START}, {SAMPLE_GUARDRAIL_STOP})，"
    f"{GUARDRAIL_SAMPLE_ROWS:,} 条确定性样本。"
)

# 默认不立即扫描全部 3603 个时间点；批量构造序列前再显式调用。
market_center = None

早期 Train 极值保护拟合完成：区间 [486, 1945)，131,072 条确定性样本。


### 6.3 旧单股票窗口（仅保留为诊断对照）

这一函数只服务人工挖洞对照，不再作为正式特征入口。正式管道在 6.5 构造完整的 `filled_num_x`；首次观测前、短缺口和长缺口都会得到有限值，并携带来源码与可信度。

In [10]:
def _ewma_update(
    current: np.ndarray | None,
    new_value: np.ndarray,
    alpha: float,
) -> np.ndarray:
    if current is None:
        return np.asarray(new_value, dtype=np.float32).copy()
    return ((1.0 - alpha) * current + alpha * new_value).astype(np.float32)


def _market_center_at(
    time_idx: int,
    numeric_values: np.ndarray,
    observed_mask: np.ndarray,
    market_center_values: np.ndarray | None,
) -> np.ndarray:
    if market_center_values is not None:
        return np.asarray(market_center_values[time_idx], dtype=np.float32)
    eligible = observed_mask[time_idx]
    if np.any(eligible):
        return np.median(
            numeric_values[time_idx, eligible], axis=0
        ).astype(np.float32)
    return np.zeros(numeric_values.shape[2], dtype=np.float32)


def build_legacy_causal_stock_window(
    stock_idx: int,
    start: int,
    stop: int,
    numeric_values: np.ndarray = num_x,
    observed_mask: np.ndarray = mask_x,
    market_center_values: np.ndarray | None = None,
    lower_guardrail: np.ndarray | None = TRAIN_LOWER,
    upper_guardrail: np.ndarray | None = TRAIN_UPPER,
    config: GapImputationConfig = GAP_CONFIG,
) -> dict[str, np.ndarray | float | int]:
    if not (0 <= start < stop <= numeric_values.shape[0]):
        raise ValueError(f"无效时间窗口：[{start}, {stop})")
    if not (0 <= stock_idx < numeric_values.shape[1]):
        raise IndexError(f"股票索引越界：{stock_idx}")

    feature_count = numeric_values.shape[2]
    window_length = stop - start
    values = np.empty((window_length, feature_count), dtype=np.float32)
    source_code = np.zeros(window_length, dtype=np.int8)
    gap_length = np.full(window_length, -1, dtype=np.int16)
    confidence = np.zeros(window_length, dtype=np.float32)
    stock_age = np.full(window_length, -1, dtype=np.int32)

    prior_real_times = np.flatnonzero(observed_mask[:start, stock_idx])
    recent_real_times = prior_real_times[-(config.drift_observations + 1):]
    last_real_time = None
    last_real_value = None
    drift = None
    first_real_time = int(prior_real_times[0]) if prior_real_times.size else None

    if recent_real_times.size:
        last_real_time = int(recent_real_times[-1])
        last_real_value = np.asarray(
            numeric_values[last_real_time, stock_idx], dtype=np.float32
        ).copy()
        for previous_time, current_time in zip(
            recent_real_times[:-1], recent_real_times[1:]
        ):
            elapsed = max(int(current_time - previous_time), 1)
            unit_change = (
                numeric_values[current_time, stock_idx]
                - numeric_values[previous_time, stock_idx]
            ) / elapsed
            drift = _ewma_update(
                drift, unit_change, config.drift_ewma_alpha
            )

    real_flags = np.zeros(window_length, dtype=bool)
    imputed_flags = np.zeros(window_length, dtype=bool)

    for local_idx, time_idx in enumerate(range(start, stop)):
        center = _market_center_at(
            time_idx, numeric_values, observed_mask, market_center_values
        )

        if observed_mask[time_idx, stock_idx]:
            current_value = np.asarray(
                numeric_values[time_idx, stock_idx], dtype=np.float32
            )
            values[local_idx] = current_value
            source_code[local_idx] = 1
            gap_length[local_idx] = 0
            confidence[local_idx] = 1.0
            real_flags[local_idx] = True

            if first_real_time is None:
                first_real_time = time_idx
            if last_real_time is not None and last_real_value is not None:
                elapsed = max(time_idx - last_real_time, 1)
                unit_change = (current_value - last_real_value) / elapsed
                drift = _ewma_update(
                    drift, unit_change, config.drift_ewma_alpha
                )
            last_real_time = time_idx
            last_real_value = current_value.copy()
        else:
            # 中性占位保持张量连续，但 source_code=0 时不能当作有效观测。
            values[local_idx] = center
            if last_real_time is not None and last_real_value is not None:
                current_gap = time_idx - last_real_time
                gap_length[local_idx] = current_gap
                if current_gap <= config.max_gap:
                    drift_value = (
                        np.zeros(feature_count, dtype=np.float32)
                        if drift is None
                        else drift
                    )
                    trend_steps = min(current_gap, config.trend_step_cap)
                    trend_value = last_real_value + trend_steps * drift_value
                    if lower_guardrail is not None and upper_guardrail is not None:
                        trend_value = np.clip(
                            trend_value, lower_guardrail, upper_guardrail
                        )
                    reliability = float(
                        np.exp(-current_gap / config.confidence_tau)
                    )
                    values[local_idx] = (
                        reliability * trend_value
                        + (1.0 - reliability) * center
                    ).astype(np.float32)
                    source_code[local_idx] = 2
                    confidence[local_idx] = reliability
                    imputed_flags[local_idx] = True

        if first_real_time is not None:
            stock_age[local_idx] = time_idx - first_real_time

    history_coverage = np.zeros(window_length, dtype=np.float32)
    for local_idx in range(window_length):
        coverage_start = max(0, local_idx - DENSE_HISTORY_LENGTH + 1)
        history_coverage[local_idx] = real_flags[coverage_start:local_idx + 1].mean()

    if not np.all(np.isfinite(values)):
        raise ValueError("历史窗口包含非有限值。")

    return {
        "time_index": np.arange(start, stop, dtype=np.int32),
        "values": values,
        "source_code": source_code,  # 0=不可用中性占位，1=真实，2=短缺口补全
        "observed_mask": real_flags,
        "imputed_mask": imputed_flags,
        "available_history_mask": real_flags | imputed_flags,
        "gap_length": gap_length,
        "confidence": confidence,
        "history_coverage": history_coverage,
        "stock_age": stock_age,
        "real_coverage": float(real_flags.mean()),
        "imputed_coverage": float(imputed_flags.mean()),
    }

### 6.4 默认全量股票与可选分层降采样

正式训练默认返回每个时间点的全部监督股票。如果硬件不足，可显式传入上限；备用方案按当期 `y1` 排名分层、按时间点固定随机种子抽样，避免继续按股票编号等间距选择。

In [11]:
def eligible_training_stocks(time_idx: int) -> np.ndarray:
    eligible = mask_y[time_idx] & np.isfinite(y1[time_idx])
    return np.flatnonzero(eligible).astype(np.int32)


def select_training_stocks(
    time_idx: int,
    cap: int | None = TRAIN_STOCK_CAP,
    strata: int = TRAIN_QUERY_STRATA,
    seed: int = SEED,
) -> np.ndarray:
    eligible = eligible_training_stocks(time_idx)
    if cap is None or eligible.size <= cap:
        return eligible
    if cap <= 0 or strata <= 0:
        raise ValueError("cap 和 strata 必须为正数。")

    labels = y1[time_idx, eligible]
    order = np.argsort(labels, kind="stable")
    ranked_bins = np.array_split(order, min(strata, cap))
    base_count, extra_count = divmod(cap, len(ranked_bins))
    rng = np.random.default_rng(seed + time_idx * 1009)
    chosen_positions = []

    for bin_idx, rank_positions in enumerate(ranked_bins):
        take = base_count + int(bin_idx < extra_count)
        selected = rng.choice(rank_positions, size=take, replace=False)
        chosen_positions.append(selected)

    chosen = eligible[np.concatenate(chosen_positions)]
    chosen.sort()
    assert chosen.size == cap
    return chosen


def equal_query_sample_weights(stock_indices: np.ndarray) -> np.ndarray:
    if stock_indices.size == 0:
        return np.empty(0, dtype=np.float32)
    # 每个时间点的权重总和为 1，避免股票数量多的时间点支配训练。
    return np.full(
        stock_indices.size, 1.0 / stock_indices.size, dtype=np.float32
    )


sample_time = TRAIN_START
sample_stocks = select_training_stocks(sample_time)
sample_weights = equal_query_sample_weights(sample_stocks)
assert sample_stocks.size == eligible_training_stocks(sample_time).size
assert np.isclose(sample_weights.sum(), 1.0)
print(f"时间点 {sample_time} 默认使用全部 {sample_stocks.size:,} 只监督股票。")

时间点 486 默认使用全部 1,753 只监督股票。


### 6.5 构造完整 filled_num_x 并自检

正式补全面板覆盖全部时间、股票和数值特征：真实值保持不变；短缺口使用真实趋势；长缺口衰减到当期市场中心；首次真实观测前使用冷启动市场中心。合成测试验证所有 X 有限及未来变化不影响过去。旧单股票函数只保留为人工挖洞对照。

In [12]:
FILLED_PANEL_SCHEMA_VERSION = 2


@dataclass
class FilledNumericPanel:
    values: np.ndarray
    source: np.ndarray
    confidence: np.ndarray
    fingerprint: str
    cache_dir: Path


def filled_panel_fingerprint(
    numeric_values: np.ndarray,
    lower_guardrail: np.ndarray,
    upper_guardrail: np.ndarray,
    config: GapImputationConfig = GAP_CONFIG,
) -> str:
    guardrail_bytes = (
        np.ascontiguousarray(lower_guardrail, dtype=np.float32).tobytes()
        + np.ascontiguousarray(upper_guardrail, dtype=np.float32).tobytes()
    )
    data_stat = DATA_PATH.stat() if DATA_PATH.exists() else None
    payload = {
        "schema_version": FILLED_PANEL_SCHEMA_VERSION,
        "shape": list(numeric_values.shape),
        "dtype": str(numeric_values.dtype),
        "gap_config": asdict(config),
        "data_source": (
            None if data_stat is None else {
                "size": int(data_stat.st_size),
                "mtime_ns": int(data_stat.st_mtime_ns),
            }
        ),
        "guardrail_sha256": hashlib.sha256(guardrail_bytes).hexdigest(),
    }
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()


def _fill_numeric_panel_arrays(
    numeric_values: np.ndarray,
    observed_mask: np.ndarray,
    lower_guardrail: np.ndarray,
    upper_guardrail: np.ndarray,
    config: GapImputationConfig = GAP_CONFIG,
    market_center_values: np.ndarray | None = None,
    value_output: np.ndarray | None = None,
    source_output: np.ndarray | None = None,
    confidence_output: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    numeric_values = np.asarray(numeric_values)
    observed_mask = np.asarray(observed_mask, dtype=bool)
    if numeric_values.ndim != 3:
        raise ValueError("numeric_values 必须为 [time, stock, feature] 三维数组。")
    time_count, stock_count, feature_count = numeric_values.shape
    if observed_mask.shape != (time_count, stock_count):
        raise ValueError("mask_x 与 numeric_values 形状不一致。")
    lower_guardrail = np.asarray(lower_guardrail, dtype=np.float32)
    upper_guardrail = np.asarray(upper_guardrail, dtype=np.float32)
    if lower_guardrail.shape != (feature_count,) or upper_guardrail.shape != (feature_count,):
        raise ValueError("补全极值保护形状错误。")
    if market_center_values is not None and np.shape(market_center_values) != (time_count, feature_count):
        raise ValueError("外部市场中心形状错误。")

    values = (
        np.empty(numeric_values.shape, dtype=np.float32)
        if value_output is None else value_output
    )
    source = (
        np.empty((time_count, stock_count), dtype=np.int8)
        if source_output is None else source_output
    )
    confidence = (
        np.empty((time_count, stock_count), dtype=np.float32)
        if confidence_output is None else confidence_output
    )
    if values.shape != numeric_values.shape:
        raise ValueError("filled_num_x 输出形状错误。")

    last_real_value = np.zeros((stock_count, feature_count), dtype=np.float32)
    previous_effective = np.zeros((stock_count, feature_count), dtype=np.float32)
    real_delta_ewma = np.zeros((stock_count, feature_count), dtype=np.float32)
    last_real_time = np.full(stock_count, -1, dtype=np.int32)
    real_count = np.zeros(stock_count, dtype=np.int32)
    previous_market_center = np.zeros(feature_count, dtype=np.float32)

    for time_idx in range(time_count):
        current_raw = np.asarray(numeric_values[time_idx], dtype=np.float32)
        current_real = np.asarray(observed_mask[time_idx], dtype=bool)
        if np.any(current_real) and not np.all(np.isfinite(current_raw[current_real])):
            raise ValueError(f"时间 {time_idx} 的真实 X 包含非有限值。")

        if market_center_values is not None:
            market_center = np.asarray(market_center_values[time_idx], dtype=np.float32)
        elif np.any(current_real):
            market_center = np.median(current_raw[current_real], axis=0).astype(np.float32)
        else:
            market_center = previous_market_center.copy()
        if not np.all(np.isfinite(market_center)):
            raise ValueError(f"时间 {time_idx} 的市场中心包含非有限值。")
        market_center = np.clip(
            market_center, lower_guardrail, upper_guardrail
        ).astype(np.float32)
        previous_market_center = market_center

        effective = np.broadcast_to(
            market_center, (stock_count, feature_count)
        ).copy()
        current_source = np.full(stock_count, 3, dtype=np.int8)
        current_confidence = np.zeros(stock_count, dtype=np.float32)
        has_real_history = real_count > 0
        gap = np.where(
            has_real_history, time_idx - last_real_time, 0
        ).astype(np.int32)

        short_gap = (~current_real) & has_real_history & (gap <= config.max_gap)
        if np.any(short_gap):
            trend_steps = np.minimum(
                gap[short_gap], config.trend_step_cap
            ).astype(np.float32)
            trend_level = (
                last_real_value[short_gap]
                + trend_steps[:, None] * real_delta_ewma[short_gap]
            )
            candidate = (
                0.5 * previous_effective[short_gap] + 0.5 * trend_level
            )
            effective[short_gap] = np.clip(
                candidate, lower_guardrail, upper_guardrail
            ).astype(np.float32)
            current_source[short_gap] = 1
            current_confidence[short_gap] = np.exp(
                -gap[short_gap] / config.confidence_tau
            ).astype(np.float32)

        long_gap = (~current_real) & has_real_history & (gap > config.max_gap)
        if np.any(long_gap):
            reliability = np.exp(
                -(gap[long_gap] - config.max_gap) / config.long_gap_tau
            ).astype(np.float32)
            effective[long_gap] = (
                reliability[:, None] * previous_effective[long_gap]
                + (1.0 - reliability[:, None]) * market_center
            ).astype(np.float32)
            effective[long_gap] = np.clip(
                effective[long_gap], lower_guardrail, upper_guardrail
            )
            current_source[long_gap] = 2
            current_confidence[long_gap] = np.exp(
                -gap[long_gap] / config.confidence_tau
            ).astype(np.float32)

        if np.any(current_real):
            effective[current_real] = current_raw[current_real]
            current_source[current_real] = 0
            current_confidence[current_real] = 1.0
        if not np.all(np.isfinite(effective)):
            raise ValueError(f"时间 {time_idx} 的 filled_num_x 包含非有限值。")

        values[time_idx] = effective
        source[time_idx] = current_source
        confidence[time_idx] = current_confidence

        returning_real = current_real & has_real_history
        if np.any(returning_real):
            elapsed = (time_idx - last_real_time[returning_real]).astype(np.float32)
            unit_delta = (
                current_raw[returning_real] - last_real_value[returning_real]
            ) / elapsed[:, None]
            first_delta = real_count[returning_real] == 1
            updated_delta = (
                (1.0 - config.drift_ewma_alpha) * real_delta_ewma[returning_real]
                + config.drift_ewma_alpha * unit_delta
            ).astype(np.float32)
            updated_delta[first_delta] = unit_delta[first_delta]
            real_delta_ewma[returning_real] = updated_delta
        if np.any(current_real):
            last_real_value[current_real] = current_raw[current_real]
            last_real_time[current_real] = time_idx
            real_count[current_real] += 1
        previous_effective = effective

    return values, source, confidence


def open_filled_numeric_panel(
    output_dir: Path,
    expected_fingerprint: str,
) -> FilledNumericPanel:
    output_dir = Path(output_dir)
    metadata = json.loads((output_dir / "metadata.json").read_text(encoding="utf-8"))
    if metadata.get("fingerprint") != expected_fingerprint:
        raise ValueError("filled_num_x 缓存指纹与当前补全配置不一致。")
    values = np.load(output_dir / "filled_num_x.npy", mmap_mode="r")
    source = np.load(output_dir / "fill_source.npy", mmap_mode="r")
    confidence = np.load(output_dir / "fill_confidence.npy", mmap_mode="r")
    if values.shape != num_x.shape or values.dtype != np.float32:
        raise ValueError(f"filled_num_x 缓存形状或类型错误：{values.shape}, {values.dtype}")
    if source.shape != mask_x.shape or confidence.shape != mask_x.shape:
        raise ValueError("补全来源或可信度缓存形状错误。")
    if not metadata.get("all_finite_verified", False):
        raise ValueError("filled_num_x 缓存缺少完整有限值验证标记。")
    return FilledNumericPanel(values, source, confidence, expected_fingerprint, output_dir)


def materialize_filled_numeric_panel(
    output_dir: Path,
    fingerprint: str,
    overwrite: bool = False,
) -> FilledNumericPanel:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    final_paths = {
        "values": output_dir / "filled_num_x.npy",
        "source": output_dir / "fill_source.npy",
        "confidence": output_dir / "fill_confidence.npy",
        "metadata": output_dir / "metadata.json",
    }
    existing = [path for path in final_paths.values() if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(f"补全缓存目录已有文件：{existing[0]}")
    partial_paths = {
        name: path.with_name(path.name + ".partial")
        for name, path in final_paths.items() if name != "metadata"
    }
    stale_partial = [path for path in partial_paths.values() if path.exists()]
    if stale_partial:
        raise FileExistsError(f"发现未完成的补全缓存：{stale_partial[0]}")

    value_map = np.lib.format.open_memmap(
        partial_paths["values"], mode="w+", dtype=np.float32, shape=num_x.shape
    )
    source_map = np.lib.format.open_memmap(
        partial_paths["source"], mode="w+", dtype=np.int8, shape=mask_x.shape
    )
    confidence_map = np.lib.format.open_memmap(
        partial_paths["confidence"], mode="w+", dtype=np.float32, shape=mask_x.shape
    )
    _fill_numeric_panel_arrays(
        num_x, mask_x, TRAIN_LOWER, TRAIN_UPPER, GAP_CONFIG,
        value_output=value_map, source_output=source_map,
        confidence_output=confidence_map,
    )
    source_counts = np.bincount(
        np.asarray(source_map).reshape(-1), minlength=4
    ).astype(np.int64)
    for array in (value_map, source_map, confidence_map):
        array.flush()
    del array, value_map, source_map, confidence_map
    for name, partial_path in partial_paths.items():
        partial_path.replace(final_paths[name])

    metadata = {
        "schema_version": FILLED_PANEL_SCHEMA_VERSION,
        "fingerprint": fingerprint,
        "shape": list(num_x.shape),
        "dtype": "float32",
        "source_counts": source_counts.tolist(),
        "source_codes": {"real": 0, "short_gap": 1, "long_gap": 2, "cold_start": 3},
        "all_finite_verified": True,
    }
    metadata_partial = final_paths["metadata"].with_suffix(".json.partial")
    metadata_partial.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    metadata_partial.replace(final_paths["metadata"])
    return open_filled_numeric_panel(output_dir, fingerprint)


def ensure_filled_numeric_panel(
    cache_root: Path = FILLED_NUMERIC_CACHE_ROOT,
    rebuild: bool = REBUILD_FILLED_NUMERIC_CACHE,
) -> FilledNumericPanel:
    fingerprint = filled_panel_fingerprint(
        num_x, TRAIN_LOWER, TRAIN_UPPER, GAP_CONFIG
    )
    output_dir = Path(cache_root) / fingerprint[:12]
    required = [
        output_dir / "filled_num_x.npy", output_dir / "fill_source.npy",
        output_dir / "fill_confidence.npy", output_dir / "metadata.json",
    ]
    complete = all(path.exists() for path in required)
    if complete and not rebuild:
        return open_filled_numeric_panel(output_dir, fingerprint)
    if not complete and any(path.exists() for path in required) and not rebuild:
        raise FileExistsError("发现不完整的 filled_num_x 缓存；请显式启用 rebuild。")
    return materialize_filled_numeric_panel(output_dir, fingerprint, overwrite=rebuild)


def run_filled_panel_self_test() -> None:
    toy_x = np.zeros((9, 3, 1), dtype=np.float32)
    toy_mask = np.zeros((9, 3), dtype=bool)
    toy_x[[2, 3, 6], 0, 0] = [10.0, 11.0, 14.0]
    toy_mask[[2, 3, 6], 0] = True
    toy_x[:, 1, 0] = np.arange(9, dtype=np.float32)
    toy_mask[:, 1] = True
    lower = np.array([-100.0], dtype=np.float32)
    upper = np.array([100.0], dtype=np.float32)
    toy_config = GapImputationConfig(max_gap=2, trend_step_cap=2, confidence_tau=4.0)
    first_values, first_source, first_confidence = _fill_numeric_panel_arrays(
        toy_x, toy_mask, lower, upper, toy_config
    )
    changed = toy_x.copy()
    changed[7:] = 9999.0
    second_values, _, _ = _fill_numeric_panel_arrays(
        changed, toy_mask, lower, upper, toy_config
    )
    assert np.all(np.isfinite(first_values))
    assert np.array_equal(first_values[toy_mask], toy_x[toy_mask])
    assert np.array_equal(first_source[:2, 0], np.full(2, 3, dtype=np.int8))
    assert np.array_equal(first_source[[2, 3, 6], 0], np.zeros(3, dtype=np.int8))
    assert np.array_equal(first_source[[4, 5], 0], np.ones(2, dtype=np.int8))
    assert np.all(first_source[:, 2] == 3)
    assert np.all((0.0 <= first_confidence) & (first_confidence <= 1.0))
    assert np.allclose(first_values[:7], second_values[:7])
    print("完整 X 因果补全面板自检通过。")


run_filled_panel_self_test()


def run_gap_imputation_self_test() -> None:
    toy_x = np.zeros((12, 2, 1), dtype=np.float32)
    toy_mask = np.zeros((12, 2), dtype=bool)
    toy_x[[2, 3, 6, 7], 0, 0] = [10.0, 11.0, 14.0, 15.0]
    toy_mask[[2, 3, 6, 7], 0] = True
    toy_x[:, 1, 0] = 0.0
    toy_mask[:, 1] = True
    original_x = toy_x.copy()
    original_mask = toy_mask.copy()

    toy_config = GapImputationConfig(
        max_gap=2,
        drift_observations=3,
        drift_ewma_alpha=0.5,
        trend_step_cap=2,
        confidence_tau=4.0,
    )
    first = build_legacy_causal_stock_window(
        0,
        0,
        8,
        numeric_values=toy_x,
        observed_mask=toy_mask,
        lower_guardrail=np.array([-100.0], dtype=np.float32),
        upper_guardrail=np.array([100.0], dtype=np.float32),
        config=toy_config,
    )
    toy_x[10:, 0, 0] = 9999.0
    second = build_legacy_causal_stock_window(
        0,
        0,
        8,
        numeric_values=toy_x,
        observed_mask=toy_mask,
        lower_guardrail=np.array([-100.0], dtype=np.float32),
        upper_guardrail=np.array([100.0], dtype=np.float32),
        config=toy_config,
    )

    assert np.array_equal(original_mask, toy_mask)
    assert np.array_equal(first["source_code"][:2], np.array([0, 0]))
    assert np.array_equal(first["source_code"][[2, 3, 6, 7]], np.ones(4))
    assert np.array_equal(first["source_code"][[4, 5]], np.full(2, 2))
    assert np.array_equal(
        first["values"][[2, 3, 6, 7], 0],
        original_x[[2, 3, 6, 7], 0, 0],
    )
    assert np.allclose(first["values"], second["values"])
    assert np.array_equal(original_x[:8], toy_x[:8])
    print("旧单股票诊断函数自检通过。")


run_gap_imputation_self_test()

完整 X 因果补全面板自检通过。
旧单股票诊断函数自检通过。


In [13]:
def _true_runs(flags: np.ndarray) -> list[tuple[int, int]]:
    padded = np.concatenate(([False], flags.astype(bool), [False])).astype(np.int8)
    changes = np.diff(padded)
    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1)
    return list(zip(starts.tolist(), stops.tolist()))


def evaluate_artificial_gaps(
    gap_lengths: tuple[int, ...] = (1, 5, 10, 30),
    samples_per_gap: int = 4,
    config: GapImputationConfig = GAP_CONFIG,
    seed: int = SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    candidate_stocks = rng.permutation(STOCK_COUNT)
    result_rows = []

    for gap in gap_lengths:
        warmup = max(config.drift_observations + 1, 12)
        required_length = warmup + gap
        completed = 0
        proposed_errors = []
        last_value_errors = []

        for stock_idx in candidate_stocks:
            train_flags = mask_x[TRAIN_START:VALID_START, stock_idx]
            suitable_runs = [
                (run_start, run_stop)
                for run_start, run_stop in _true_runs(train_flags)
                if run_stop - run_start >= required_length
            ]
            if not suitable_runs:
                continue

            run_start, run_stop = suitable_runs[-1]
            local_stop = TRAIN_START + run_stop
            local_start = local_stop - required_length
            local_x = np.asarray(
                num_x[local_start:local_stop, stock_idx:stock_idx + 1],
                dtype=np.float32,
            ).copy()
            local_mask = np.ones((required_length, 1), dtype=bool)
            true_hidden = local_x[-gap:, 0].copy()
            local_mask[-gap:, 0] = False

            center_slice = compute_cross_sectional_median(
                num_x, mask_x, local_start, local_stop
            )
            reconstructed = build_legacy_causal_stock_window(
                0,
                0,
                required_length,
                numeric_values=local_x,
                observed_mask=local_mask,
                market_center_values=center_slice,
                lower_guardrail=TRAIN_LOWER,
                upper_guardrail=TRAIN_UPPER,
                config=config,
            )
            proposed = reconstructed["values"][-gap:]
            last_value = np.repeat(local_x[-gap - 1, 0][None, :], gap, axis=0)
            proposed_errors.append((proposed - true_hidden).ravel())
            last_value_errors.append((last_value - true_hidden).ravel())
            completed += 1
            if completed >= samples_per_gap:
                break

        if completed == 0:
            result_rows.append(
                {"gap": gap, "samples": 0, "method": "unavailable"}
            )
            continue

        proposed_error = np.concatenate(proposed_errors)
        last_value_error = np.concatenate(last_value_errors)
        for method, error in (
            ("gap_aware", proposed_error),
            ("last_value", last_value_error),
        ):
            result_rows.append(
                {
                    "gap": gap,
                    "samples": completed,
                    "method": method,
                    "mae": float(np.mean(np.abs(error))),
                    "rmse": float(np.sqrt(np.mean(error ** 2))),
                }
            )

    return pd.DataFrame(result_rows)


artificial_gap_report = (
    evaluate_artificial_gaps() if RUN_ARTIFICIAL_GAP_CHECK else None
)
if artificial_gap_report is not None:
    display(artificial_gap_report)

### 6.6 第一部分的选择结果与下游接口

第一部分完成后，`filled_num_x`、`fill_source`、`fill_confidence` 与缓存指纹写入 `SELECTED_SAMPLE_CONFIG`；后续特征只接受这套补全结果。

In [14]:
FILLED_NUMERIC_PANEL = ensure_filled_numeric_panel()
filled_num_x = FILLED_NUMERIC_PANEL.values
fill_source = FILLED_NUMERIC_PANEL.source
fill_confidence = FILLED_NUMERIC_PANEL.confidence
filled_metadata = json.loads(
    (FILLED_NUMERIC_PANEL.cache_dir / "metadata.json").read_text(encoding="utf-8")
)

SELECTED_SAMPLE_CONFIG = {
    "gap_config": GAP_CONFIG,
    "stock_cap": TRAIN_STOCK_CAP,
    "query_weight_mode": "equal_total_weight_per_time",
    "dense_history_length": DENSE_HISTORY_LENGTH,
    "long_history_lags": LONG_HISTORY_LAGS,
    "guardrail_fit_interval": (TRAIN_START, SAMPLE_GUARDRAIL_STOP),
    "lower_guardrail": TRAIN_LOWER,
    "upper_guardrail": TRAIN_UPPER,
    "filled_panel_fingerprint": FILLED_NUMERIC_PANEL.fingerprint,
    "filled_panel_cache_dir": FILLED_NUMERIC_PANEL.cache_dir,
    "sample_gate": "mask_y",
}
print(
    "第一部分已选择：完整 X 因果补全、mask_y 样本口径、每时点等总权重、"
    f"股票上限={SELECTED_SAMPLE_CONFIG['stock_cap']}；"
    f"补全来源计数={filled_metadata['source_counts']}。"
)

第一部分已选择：完整 X 因果补全、mask_y 样本口径、每时点等总权重、股票上限=None；补全来源计数=[11529312, 0, 0, 7501734]。


## 7. 第二部分：双重相对坐标数值特征

本节保留全部 99 个补全后数值特征，并建立三类互补表示：Train-only 稳健截尾后的 `filled_num_x`、`mask_y` 股票池横截面排名，以及只依赖过去补全序列的多尺度惊奇值。前 20 个稳定特征只限制动态派生特征规模。

In [15]:
@dataclass(frozen=True)
class NumericFeatureConfig:
    stable_feature_count: int = 20
    discovery_fraction: float = 0.60
    discovery_time_samples: int = 128
    check_time_samples: int = 64
    correlation_stocks_per_time: int = 512
    max_abs_feature_correlation: float = 0.92
    ewma_half_lives: tuple[int, ...] = (5, 20, 60)
    surprise_clip: float = 5.0
    minimum_history_observations: int = 3
    history_confidence_tau: float = 10.0
    state_epsilon: float = 1e-3


NUMERIC_CONFIG = NumericFeatureConfig()
FEATURE_DISCOVERY_START = TRAIN_START
FEATURE_DISCOVERY_STOP = TRAIN_START + int(
    round((VALID_START - TRAIN_START) * NUMERIC_CONFIG.discovery_fraction)
)
FEATURE_CHECK_START = FEATURE_DISCOVERY_STOP
FEATURE_CHECK_STOP = VALID_START
NUMERIC_CACHE_ROOT = CACHE_DIR / "numeric_feature_bank_v2"
RUN_NUMERIC_DISCOVERY = True
BUILD_NUMERIC_FEATURE_CACHE = False

# 如果暂时关闭重新发现，使用当前最佳历史实验的前 20 个稳定特征。
HISTORICAL_STABLE_FEATURES = np.array(
    [8, 11, 57, 41, 90, 68, 39, 40, 73, 47,
     53, 72, 48, 74, 86, 38, 50, 3, 42, 71],
    dtype=np.int32,
)

pd.Series(
    {
        "discovery_interval": (FEATURE_DISCOVERY_START, FEATURE_DISCOVERY_STOP),
        "internal_check_interval": (FEATURE_CHECK_START, FEATURE_CHECK_STOP),
        "stable_feature_count": NUMERIC_CONFIG.stable_feature_count,
        "ewma_half_lives": NUMERIC_CONFIG.ewma_half_lives,
        "full_cache_enabled": BUILD_NUMERIC_FEATURE_CACHE,
    },
    name="numeric_feature_config",
)

discovery_interval          (486, 1945)
internal_check_interval    (1945, 2918)
stable_feature_count                 20
ewma_half_lives             (5, 20, 60)
full_cache_enabled                False
Name: numeric_feature_config, dtype: object

### 7.1 只在较早 Train 中发现稳定动态特征

特征发现只读取 `[FEATURE_DISCOVERY_START, FEATURE_DISCOVERY_STOP)` 的标签；较后的 Train 子区间只用于检查，不参与选择，官方 Valid 完全保留。评分同时考虑平均 RankIC、时间稳定性和冗余相关性。

In [16]:
def _rank_ic_vector(features: np.ndarray, target: np.ndarray) -> np.ndarray:
    if features.ndim != 2 or target.ndim != 1:
        raise ValueError("features 必须为二维，target 必须为一维。")
    if features.shape[0] != target.size:
        raise ValueError("特征行数与标签长度不一致。")
    if target.size < 2:
        return np.full(features.shape[1], np.nan, dtype=np.float64)

    feature_ranks = rankdata(features, axis=0, method="average")
    target_ranks = rankdata(target, method="average")
    feature_centered = feature_ranks - feature_ranks.mean(axis=0, keepdims=True)
    target_centered = target_ranks - target_ranks.mean()
    numerator = feature_centered.T @ target_centered
    denominator = np.sqrt(
        np.sum(feature_centered ** 2, axis=0)
        * np.sum(target_centered ** 2)
    )
    return np.divide(
        numerator,
        denominator,
        out=np.full(features.shape[1], np.nan, dtype=np.float64),
        where=denominator > 0,
    )


def sample_time_rank_ic(
    start: int,
    stop: int,
    time_samples: int,
    feature_indices: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    feature_indices = (
        np.arange(NUMERIC_FEATURE_COUNT, dtype=np.int32)
        if feature_indices is None
        else np.asarray(feature_indices, dtype=np.int32)
    )
    count = min(time_samples, stop - start)
    sampled_times = np.unique(
        np.linspace(start, stop - 1, count, dtype=np.int64)
    )
    ic_rows = []
    used_times = []

    for time_idx in sampled_times:
        stocks = eligible_training_stocks(int(time_idx))
        if stocks.size < 20:
            continue
        current_features = np.asarray(
            filled_num_x[time_idx, stocks][:, feature_indices], dtype=np.float32
        )
        current_target = np.asarray(y1[time_idx, stocks], dtype=np.float32)
        ic_rows.append(_rank_ic_vector(current_features, current_target))
        used_times.append(int(time_idx))

    if not ic_rows:
        raise RuntimeError("指定时间区间没有足够样本计算 RankIC。")
    return (
        np.asarray(ic_rows, dtype=np.float64),
        np.asarray(used_times, dtype=np.int64),
    )


def _sample_discovery_feature_rows(
    sampled_times: np.ndarray,
    lower: np.ndarray,
    upper: np.ndarray,
    config: NumericFeatureConfig = NUMERIC_CONFIG,
    seed: int = SEED,
) -> np.ndarray:
    rows = []
    for time_idx in sampled_times:
        stocks = eligible_training_stocks(int(time_idx))
        if stocks.size > config.correlation_stocks_per_time:
            rng = np.random.default_rng(seed + int(time_idx) * 1009)
            stocks = np.sort(
                rng.choice(
                    stocks,
                    size=config.correlation_stocks_per_time,
                    replace=False,
                )
            )
        if stocks.size:
            rows.append(
                np.clip(filled_num_x[time_idx, stocks], lower, upper).astype(np.float32)
            )
    if not rows:
        raise RuntimeError("无法抽取用于冗余检查的数值样本。")
    return np.concatenate(rows, axis=0)


def discover_stable_numeric_features(
    lower: np.ndarray,
    upper: np.ndarray,
    config: NumericFeatureConfig = NUMERIC_CONFIG,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    ic_matrix, sampled_times = sample_time_rank_ic(
        FEATURE_DISCOVERY_START,
        FEATURE_DISCOVERY_STOP,
        config.discovery_time_samples,
    )
    mean_ic = np.nanmean(ic_matrix, axis=0)
    std_ic = np.nanstd(ic_matrix, axis=0)
    positive_share = np.nanmean(ic_matrix >= 0, axis=0)
    sign_stability = np.maximum(positive_share, 1.0 - positive_share)
    negative_time_share = np.where(
        mean_ic >= 0,
        np.nanmean(ic_matrix < 0, axis=0),
        np.nanmean(ic_matrix > 0, axis=0),
    )
    stability_score = (
        np.abs(mean_ic) * sign_stability / (std_ic + 0.01)
    )

    correlation_rows = _sample_discovery_feature_rows(
        sampled_times, lower, upper, config
    )
    correlation = np.corrcoef(correlation_rows, rowvar=False)
    correlation = np.nan_to_num(correlation, nan=0.0)

    candidate_order = np.argsort(-stability_score, kind="stable")
    selected = []
    for feature_idx in candidate_order:
        if selected and np.max(np.abs(correlation[feature_idx, selected])) >= (
            config.max_abs_feature_correlation
        ):
            continue
        selected.append(int(feature_idx))
        if len(selected) >= config.stable_feature_count:
            break
    if len(selected) < config.stable_feature_count:
        selected.extend(
            int(index) for index in candidate_order if int(index) not in selected
        )
    selected = np.asarray(
        selected[:config.stable_feature_count], dtype=np.int32
    )

    diagnostics = pd.DataFrame(
        {
            "feature_index": np.arange(NUMERIC_FEATURE_COUNT),
            "feature": [f"num_{idx}" for idx in range(NUMERIC_FEATURE_COUNT)],
            "discovery_mean_rank_ic": mean_ic,
            "discovery_std_rank_ic": std_ic,
            "sign_stability": sign_stability,
            "opposite_sign_time_share": negative_time_share,
            "stability_score": stability_score,
            "selected_for_dynamic_views": False,
        }
    )
    diagnostics.loc[
        diagnostics["feature_index"].isin(selected),
        "selected_for_dynamic_views",
    ] = True
    return diagnostics, selected, correlation


DISCOVERY_LOWER, DISCOVERY_UPPER, DISCOVERY_GUARDRAIL_ROWS = (
    fit_train_guardrails(
        num_x, mask_x, FEATURE_DISCOVERY_START, FEATURE_DISCOVERY_STOP
    )
)

if RUN_NUMERIC_DISCOVERY:
    numeric_feature_diagnostics, STABLE_NUMERIC_FEATURES, DISCOVERY_CORRELATION = (
        discover_stable_numeric_features(DISCOVERY_LOWER, DISCOVERY_UPPER)
    )
else:
    STABLE_NUMERIC_FEATURES = HISTORICAL_STABLE_FEATURES.copy()
    numeric_feature_diagnostics = pd.DataFrame(
        {
            "feature_index": STABLE_NUMERIC_FEATURES,
            "feature": [f"num_{idx}" for idx in STABLE_NUMERIC_FEATURES],
            "discovery_mean_rank_ic": np.nan,
            "discovery_std_rank_ic": np.nan,
            "sign_stability": np.nan,
            "opposite_sign_time_share": np.nan,
            "stability_score": np.nan,
            "selected_for_dynamic_views": True,
        }
    )
    DISCOVERY_CORRELATION = np.empty((0, 0), dtype=np.float32)

check_ic_matrix, _ = sample_time_rank_ic(
    FEATURE_CHECK_START,
    FEATURE_CHECK_STOP,
    NUMERIC_CONFIG.check_time_samples,
    STABLE_NUMERIC_FEATURES,
)
internal_check_summary = pd.DataFrame(
    {
        "feature_index": STABLE_NUMERIC_FEATURES,
        "internal_check_mean_rank_ic": np.nanmean(check_ic_matrix, axis=0),
        "internal_check_std_rank_ic": np.nanstd(check_ic_matrix, axis=0),
    }
)

selected_feature_summary = (
    numeric_feature_diagnostics[
        numeric_feature_diagnostics["feature_index"].isin(STABLE_NUMERIC_FEATURES)
    ]
    .merge(internal_check_summary, on="feature_index", how="left")
    .sort_values("stability_score", ascending=False, na_position="last")
)
print("动态派生特征索引：", STABLE_NUMERIC_FEATURES.tolist())
selected_feature_summary

动态派生特征索引： [90, 57, 40, 8, 82, 42, 11, 55, 48, 47, 91, 74, 39, 7, 68, 41, 56, 69, 33, 83]


,feature_index,feature,discovery_mean_rank_ic,discovery_std_rank_ic,sign_stability,opposite_sign_time_share,stability_score,selected_for_dynamic_views,internal_check_mean_rank_ic,internal_check_std_rank_ic
18,90,num_90,-0.062800,0.065957,0.828125,0.171875,0.684685,True,-0.037101,0.085141
12,57,num_57,0.052150,0.054768,0.812500,0.187500,0.654210,True,0.057860,0.051065
5,40,num_40,-0.058576,0.068506,0.796875,0.203125,0.594578,True,-0.029622,0.059351
1,8,num_8,-0.090305,0.125669,0.789062,0.210938,0.525218,True,-0.030175,0.135195
16,82,num_82,-0.027828,0.035763,0.820312,0.179688,0.498819,True,-0.010366,0.041508
7,42,num_42,-0.051388,0.070508,0.773438,0.226562,0.493678,True,-0.019772,0.050741
2,11,num_11,-0.069110,0.102380,0.773438,0.226562,0.475640,True,-0.039238,0.105096
10,55,num_55,-0.044605,0.062252,0.757812,0.242188,0.467838,True,-0.028531,0.051746
9,48,num_48,-0.040501,0.059061,0.734375,0.265625,0.430683,True,-0.027838,0.068597
8,47,num_47,-0.038021,0.070470,0.734375,0.265625,0.346985,True,-0.032648,0.060019


### 7.2 定义横截面排名与特征块

横截面排名位于 `[-1, 1]`，在当期全部 `mask_y=True` 股票的 `filled_num_x` 上计算。所有特征块保持独立切片，后续可以按照 `raw → rank → surprise → surprise_rank → trend` 的顺序做消融。

In [17]:
def signed_rank_columns(
    values: np.ndarray,
    valid_rows: np.ndarray | None = None,
) -> np.ndarray:
    values = np.asarray(values)
    if values.ndim != 2:
        raise ValueError("values 必须为二维矩阵。")
    output = np.zeros(values.shape, dtype=np.float32)
    valid_rows = (
        np.ones(values.shape[0], dtype=bool)
        if valid_rows is None
        else np.asarray(valid_rows, dtype=bool)
    )
    valid_count = int(np.count_nonzero(valid_rows))
    if valid_count <= 1:
        return output
    ranks = rankdata(values[valid_rows], axis=0, method="average")
    output[valid_rows] = (
        2.0 * (ranks - 1.0) / (valid_count - 1.0) - 1.0
    ).astype(np.float32)
    return output


def numeric_feature_layout(
    stable_features: np.ndarray = STABLE_NUMERIC_FEATURES,
    config: NumericFeatureConfig = NUMERIC_CONFIG,
    raw_feature_count: int = NUMERIC_FEATURE_COUNT,
) -> tuple[list[str], dict[str, slice]]:
    stable_features = np.asarray(stable_features, dtype=np.int32)
    names = []
    blocks = {}

    def add_block(block_name: str, block_names: list[str]) -> None:
        start = len(names)
        names.extend(block_names)
        blocks[block_name] = slice(start, len(names))

    add_block(
        "raw", [f"filled_raw_clip_num_{idx}" for idx in range(raw_feature_count)]
    )
    add_block(
        "cross_section_rank",
        [f"filled_cs_rank_num_{idx}" for idx in range(raw_feature_count)],
    )
    add_block(
        "surprise",
        [
            f"filled_surprise_h{half_life}_num_{feature_idx}"
            for half_life in config.ewma_half_lives
            for feature_idx in stable_features
        ],
    )
    add_block(
        "surprise_rank",
        [
            f"filled_surprise_rank_h{half_life}_num_{feature_idx}"
            for half_life in config.ewma_half_lives
            for feature_idx in stable_features
        ],
    )
    add_block(
        "trend",
        [f"filled_fast_trend_num_{feature_idx}" for feature_idx in stable_features]
        + [f"filled_slow_trend_num_{feature_idx}" for feature_idx in stable_features],
    )
    add_block(
        "history_state",
        [
            "current_fill_confidence",
            "current_fill_source_scaled",
            "missing_gap_before_current",
            "real_coverage_30",
            "imputed_coverage_30",
        ],
    )
    return names, blocks


NUMERIC_FEATURE_NAMES, NUMERIC_FEATURE_BLOCKS = numeric_feature_layout()
numeric_fingerprint_payload = {
    "schema_version": 2,
    "filled_panel_fingerprint": FILLED_NUMERIC_PANEL.fingerprint,
    "numeric_config": asdict(NUMERIC_CONFIG),
    "stable_features": STABLE_NUMERIC_FEATURES.tolist(),
    "feature_names": NUMERIC_FEATURE_NAMES,
    "dense_history_length": DENSE_HISTORY_LENGTH,
}
NUMERIC_FEATURE_FINGERPRINT = hashlib.sha256(
    json.dumps(
        numeric_fingerprint_payload, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
).hexdigest()
NUMERIC_CACHE_DIR = NUMERIC_CACHE_ROOT / NUMERIC_FEATURE_FINGERPRINT[:12]
numeric_layout_summary = pd.DataFrame(
    [
        {
            "block": block_name,
            "start": block_slice.start,
            "stop": block_slice.stop,
            "feature_count": block_slice.stop - block_slice.start,
        }
        for block_name, block_slice in NUMERIC_FEATURE_BLOCKS.items()
    ]
)
numeric_layout_summary

,block,start,stop,feature_count
0,raw,0,99,99
1,cross_section_rank,99,198,99
2,surprise,198,258,60
3,surprise_rank,258,318,60
4,trend,318,358,40
5,history_state,358,363,5


### 7.3 流式生成因果数值特征

生成器从时间 0 顺序读取 `filled_num_x` 并更新每只股票的连续 EWMA 状态，但只在指定区间、按照 `mask_y` 输出行。当前 `t` 的惊奇值使用截至 `t-1` 的状态；完成输出后才用补全后的 `x_t` 更新状态。`mask_x` 只生成来源与可信度特征。

In [18]:
def _general_target_stocks(
    time_idx: int,
    role: str,
    label_mask: np.ndarray,
    targets: np.ndarray,
    stock_cap: int | None,
    seed: int,
) -> np.ndarray:
    eligible = np.asarray(label_mask[time_idx], dtype=bool).copy()
    if role != "test":
        eligible &= np.isfinite(targets[time_idx])
    stocks = np.flatnonzero(eligible).astype(np.int32)
    if role != "train" or stock_cap is None or stocks.size <= stock_cap:
        return stocks

    labels = targets[time_idx, stocks]
    order = np.argsort(labels, kind="stable")
    bins = np.array_split(order, min(TRAIN_QUERY_STRATA, stock_cap))
    base_count, extra_count = divmod(stock_cap, len(bins))
    rng = np.random.default_rng(seed + time_idx * 1009)
    chosen = []
    for bin_idx, positions in enumerate(bins):
        take = base_count + int(bin_idx < extra_count)
        chosen.append(rng.choice(positions, size=take, replace=False))
    selected = stocks[np.concatenate(chosen)]
    selected.sort()
    return selected


def iter_causal_numeric_features(
    start: int,
    stop: int,
    role: str,
    stable_features: np.ndarray = STABLE_NUMERIC_FEATURES,
    lower_guardrail: np.ndarray = TRAIN_LOWER,
    upper_guardrail: np.ndarray = TRAIN_UPPER,
    stock_cap: int | None = TRAIN_STOCK_CAP,
    config: NumericFeatureConfig = NUMERIC_CONFIG,
    numeric_values: np.ndarray = filled_num_x,
    fill_source_values: np.ndarray = fill_source,
    fill_confidence_values: np.ndarray = fill_confidence,
    observed_mask: np.ndarray = mask_x,
    label_mask: np.ndarray = mask_y,
    targets: np.ndarray = y1,
    seed: int = SEED,
):
    if role not in {"train", "valid", "test"}:
        raise ValueError(f"未知 role：{role}")
    if not (0 <= start < stop <= numeric_values.shape[0]):
        raise ValueError(f"无效时间区间：[{start}, {stop})")
    if np.shape(fill_source_values) != np.shape(observed_mask):
        raise ValueError("fill_source 与 mask_x 形状不一致。")
    if np.shape(fill_confidence_values) != np.shape(observed_mask):
        raise ValueError("fill_confidence 与 mask_x 形状不一致。")

    stable_features = np.asarray(stable_features, dtype=np.int32)
    half_lives = np.asarray(config.ewma_half_lives, dtype=np.float32)
    if half_lives.size != 3 or np.any(half_lives <= 0):
        raise ValueError("当前趋势定义要求 3 个正数 EWMA 半衰期。")
    if np.any(stable_features < 0) or np.any(
        stable_features >= numeric_values.shape[2]
    ):
        raise ValueError("stable_features 包含越界索引。")
    active_feature_names, _ = numeric_feature_layout(
        stable_features, config, numeric_values.shape[2]
    )
    stock_count = numeric_values.shape[1]
    stable_count = stable_features.size
    state_mean = np.zeros(
        (half_lives.size, stock_count, stable_count), dtype=np.float32
    )
    initial_scale = np.maximum(
        (upper_guardrail[stable_features] - lower_guardrail[stable_features]) / 6.0,
        config.state_epsilon,
    ).astype(np.float32)
    state_abs_deviation = np.broadcast_to(
        initial_scale, state_mean.shape
    ).copy()
    last_real_time = np.full(stock_count, -1, dtype=np.int32)
    first_real_time = np.full(stock_count, -1, dtype=np.int32)
    observation_count = np.zeros(stock_count, dtype=np.int32)
    real_coverage_ring = np.zeros((DENSE_HISTORY_LENGTH, stock_count), dtype=bool)
    imputed_coverage_ring = np.zeros((DENSE_HISTORY_LENGTH, stock_count), dtype=bool)
    real_coverage_count = np.zeros(stock_count, dtype=np.int16)
    imputed_coverage_count = np.zeros(stock_count, dtype=np.int16)
    effective_state_count = np.zeros(stock_count, dtype=np.int32)

    for time_idx in range(stop):
        current_observed = np.asarray(observed_mask[time_idx], dtype=bool)
        ring_slot = time_idx % DENSE_HISTORY_LENGTH
        current_imputed = ~current_observed
        real_coverage_count -= real_coverage_ring[ring_slot].astype(np.int16)
        imputed_coverage_count -= imputed_coverage_ring[ring_slot].astype(np.int16)
        real_coverage_ring[ring_slot] = current_observed
        imputed_coverage_ring[ring_slot] = current_imputed
        real_coverage_count += current_observed.astype(np.int16)
        imputed_coverage_count += current_imputed.astype(np.int16)

        observed_stocks = np.flatnonzero(current_observed).astype(np.int32)
        ranking_stocks = np.flatnonzero(label_mask[time_idx]).astype(np.int32)

        if ranking_stocks.size:
            current_raw = np.asarray(
                numeric_values[time_idx, ranking_stocks], dtype=np.float32
            )
            clipped_raw = np.clip(
                current_raw, lower_guardrail, upper_guardrail
            ).astype(np.float32)
            cross_section_rank = signed_rank_columns(current_raw)

            prior_count = effective_state_count[ranking_stocks]
            has_prior_history = (
                prior_count >= config.minimum_history_observations
            )
            missing_gap = np.where(
                last_real_time[ranking_stocks] >= 0,
                time_idx - last_real_time[ranking_stocks] - 1,
                -1,
            ).astype(np.int32)
            history_confidence = np.where(
                has_prior_history & (last_real_time[ranking_stocks] >= 0),
                np.exp(
                    -np.maximum(missing_gap, 0) / config.history_confidence_tau
                ),
                0.0,
            ).astype(np.float32)
            stable_current = clipped_raw[:, stable_features]

            surprise_blocks = []
            surprise_rank_blocks = []
            for half_idx in range(half_lives.size):
                means = state_mean[half_idx, ranking_stocks]
                scales = np.maximum(
                    state_abs_deviation[half_idx, ranking_stocks],
                    config.state_epsilon,
                )
                surprise = np.clip(
                    (stable_current - means) / scales,
                    -config.surprise_clip,
                    config.surprise_clip,
                ).astype(np.float32)
                surprise[~has_prior_history] = 0.0
                surprise *= history_confidence[:, None]
                surprise_blocks.append(surprise)
                surprise_rank_blocks.append(
                    signed_rank_columns(surprise, has_prior_history)
                )

            fast_trend = np.clip(
                (
                    state_mean[0, ranking_stocks]
                    - state_mean[1, ranking_stocks]
                ),
                -config.surprise_clip,
                config.surprise_clip,
            ).astype(np.float32)
            slow_trend = np.clip(
                (
                    state_mean[1, ranking_stocks]
                    - state_mean[2, ranking_stocks]
                ),
                -config.surprise_clip,
                config.surprise_clip,
            ).astype(np.float32)
            fast_trend *= history_confidence[:, None]
            slow_trend *= history_confidence[:, None]

            if time_idx >= start:
                target_stocks = _general_target_stocks(
                    time_idx,
                    role,
                    label_mask,
                    targets,
                    stock_cap,
                    seed,
                )
                position_lookup = np.full(stock_count, -1, dtype=np.int32)
                position_lookup[ranking_stocks] = np.arange(
                    ranking_stocks.size, dtype=np.int32
                )
                target_positions = position_lookup[target_stocks]
                if np.any(target_positions < 0):
                    raise AssertionError("目标股票不在当期排名股票池中。")

                current_source = np.asarray(
                    fill_source_values[time_idx, target_stocks], dtype=np.float32
                )
                current_confidence = np.asarray(
                    fill_confidence_values[time_idx, target_stocks], dtype=np.float32
                )
                history_state = np.column_stack(
                    [
                        current_confidence,
                        current_source / 3.0,
                        np.maximum(missing_gap[target_positions], 0),
                        real_coverage_count[target_stocks] / DENSE_HISTORY_LENGTH,
                        imputed_coverage_count[target_stocks] / DENSE_HISTORY_LENGTH,
                    ]
                ).astype(np.float32)
                matrix = np.column_stack(
                    [
                        clipped_raw[target_positions],
                        cross_section_rank[target_positions],
                        *[block[target_positions] for block in surprise_blocks],
                        *[block[target_positions] for block in surprise_rank_blocks],
                        fast_trend[target_positions],
                        slow_trend[target_positions],
                        history_state,
                    ]
                ).astype(np.float32)
                if matrix.shape[1] != len(active_feature_names):
                    raise AssertionError(
                        f"特征列数错误：{matrix.shape[1]} != {len(active_feature_names)}"
                    )
                if not np.all(np.isfinite(matrix)):
                    raise ValueError("数值特征矩阵包含非有限值。")
                yield {
                    "time_idx": time_idx,
                    "stocks": target_stocks,
                    "features": matrix,
                    "labels": (
                        None
                        if role == "test"
                        else np.asarray(targets[time_idx, target_stocks], dtype=np.float32)
                    ),
                    "query_weights": equal_query_sample_weights(target_stocks),
                }

        # filled_num_x 已完整，所有股票都连续更新状态；更新发生在 t 输出之后。
        if True:
            current_stable = np.clip(
                numeric_values[time_idx][:, stable_features],
                lower_guardrail[stable_features],
                upper_guardrail[stable_features],
            ).astype(np.float32)
            old_count = effective_state_count
            elapsed = np.ones(stock_count, dtype=np.float32)
            first_observation = old_count == 0

            for half_idx, half_life in enumerate(half_lives):
                decay = np.power(0.5, elapsed / half_life).astype(np.float32)
                previous_mean = state_mean[half_idx].copy()
                previous_scale = state_abs_deviation[half_idx].copy()
                new_mean = (
                    decay[:, None] * previous_mean
                    + (1.0 - decay[:, None]) * current_stable
                )
                new_scale = (
                    decay[:, None] * previous_scale
                    + (1.0 - decay[:, None])
                    * np.abs(current_stable - previous_mean)
                )
                new_mean[first_observation] = current_stable[first_observation]
                new_scale[first_observation] = initial_scale
                state_mean[half_idx] = new_mean.astype(np.float32)
                state_abs_deviation[half_idx] = np.maximum(
                    new_scale, config.state_epsilon
                ).astype(np.float32)

            effective_state_count += 1
            if observed_stocks.size:
                first_real_observation = observation_count[observed_stocks] == 0
                first_real_time[observed_stocks[first_real_observation]] = time_idx
                last_real_time[observed_stocks] = time_idx
                observation_count[observed_stocks] += 1

### 7.4 可选的全量特征缓存

全量特征按需写入 `.npy` memmap，并保存时间、股票、标签、组大小、补全指纹和特征指纹。Train 与 Valid 共用一份缓存，内部走步验证和最终训练只切片读取，避免重复保存。

In [19]:
def materialize_numeric_feature_cache(
    start: int,
    stop: int,
    role: str,
    output_dir: Path,
    stock_cap: int | None = TRAIN_STOCK_CAP,
    overwrite: bool = False,
) -> dict:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    final_paths = {
        "features": output_dir / "features.npy",
        "time_index": output_dir / "time_index.npy",
        "stock_index": output_dir / "stock_index.npy",
        "query_weights": output_dir / "query_weights.npy",
        "group_sizes": output_dir / "group_sizes.npy",
        "labels": output_dir / "labels.npy",
        "metadata": output_dir / "metadata.json",
    }
    existing = [path for path in final_paths.values() if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(
            f"缓存目录已有文件；如需替换请显式设置 overwrite=True：{existing[0]}"
        )

    group_sizes = np.array(
        [
            _general_target_stocks(
                time_idx, role, mask_y, y1, stock_cap, SEED
            ).size
            for time_idx in range(start, stop)
        ],
        dtype=np.int32,
    )
    row_count = int(group_sizes.sum())
    feature_count = len(NUMERIC_FEATURE_NAMES)
    temporary_paths = {
        name: path.with_name(path.name + ".partial")
        for name, path in final_paths.items()
        if name != "metadata"
    }
    stale_partial = [path for path in temporary_paths.values() if path.exists()]
    if stale_partial:
        raise FileExistsError(f"发现未完成的临时缓存：{stale_partial[0]}")

    feature_map = np.lib.format.open_memmap(
        temporary_paths["features"],
        mode="w+",
        dtype=np.float32,
        shape=(row_count, feature_count),
    )
    time_map = np.lib.format.open_memmap(
        temporary_paths["time_index"], mode="w+", dtype=np.int32, shape=(row_count,)
    )
    stock_map = np.lib.format.open_memmap(
        temporary_paths["stock_index"], mode="w+", dtype=np.int32, shape=(row_count,)
    )
    weight_map = np.lib.format.open_memmap(
        temporary_paths["query_weights"], mode="w+", dtype=np.float32, shape=(row_count,)
    )
    label_map = None
    if role != "test":
        label_map = np.lib.format.open_memmap(
            temporary_paths["labels"], mode="w+", dtype=np.float32, shape=(row_count,)
        )

    offset = 0
    for batch in iter_causal_numeric_features(
        start, stop, role, stock_cap=stock_cap
    ):
        batch_size = batch["stocks"].size
        next_offset = offset + batch_size
        feature_map[offset:next_offset] = batch["features"]
        time_map[offset:next_offset] = batch["time_idx"]
        stock_map[offset:next_offset] = batch["stocks"]
        weight_map[offset:next_offset] = batch["query_weights"]
        if label_map is not None:
            label_map[offset:next_offset] = batch["labels"]
        offset = next_offset
    if offset != row_count:
        raise AssertionError(f"缓存行数错误：{offset} != {row_count}")

    with temporary_paths["group_sizes"].open("wb") as handle:
        np.save(handle, group_sizes)
    for array in (feature_map, time_map, stock_map, weight_map, label_map):
        if array is not None:
            array.flush()
    del array
    del feature_map, time_map, stock_map, weight_map, label_map

    for name, temporary_path in temporary_paths.items():
        if name == "labels" and role == "test":
            continue
        temporary_path.replace(final_paths[name])

    metadata = {
        "role": role,
        "start": start,
        "stop": stop,
        "rows": row_count,
        "feature_count": feature_count,
        "stock_cap": stock_cap,
        "stable_numeric_features": STABLE_NUMERIC_FEATURES.tolist(),
        "feature_names": NUMERIC_FEATURE_NAMES,
        "fingerprint": NUMERIC_FEATURE_FINGERPRINT,
        "filled_panel_fingerprint": FILLED_NUMERIC_PANEL.fingerprint,
        "feature_blocks": {
            name: [block.start, block.stop]
            for name, block in NUMERIC_FEATURE_BLOCKS.items()
        },
    }
    metadata_partial = final_paths["metadata"].with_suffix(".json.partial")
    metadata_partial.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    metadata_partial.replace(final_paths["metadata"])
    return metadata


numeric_cache_metadata = None
if BUILD_NUMERIC_FEATURE_CACHE:
    numeric_cache_metadata = materialize_numeric_feature_cache(
        TRAIN_START,
        TEST_START,
        "train",
        NUMERIC_CACHE_DIR / "train_valid",
    )

### 7.5 数值处理自检

合成测试检查排名范围、特征块列数、全量股票默认行为、矩阵有限性以及未来数据变化不影响较早时间输出。

In [20]:
def run_numeric_feature_self_test() -> None:
    toy_rng = np.random.default_rng(123)
    toy_numeric = toy_rng.normal(size=(10, 6, 4)).astype(np.float32)
    toy_mask_x = np.ones((10, 6), dtype=bool)
    toy_mask_y = np.ones((10, 6), dtype=bool)
    toy_targets = np.tile(
        np.linspace(0.0, 1.0, 6, dtype=np.float32), (10, 1)
    )
    toy_mask_x[4, 0] = False
    toy_numeric[4, 0] = 0.0
    toy_lower = np.full(4, -4.0, dtype=np.float32)
    toy_upper = np.full(4, 4.0, dtype=np.float32)
    toy_stable = np.array([0, 2], dtype=np.int32)
    toy_config = NumericFeatureConfig(
        stable_feature_count=2,
        ewma_half_lives=(2, 4, 6),
        minimum_history_observations=2,
    )
    toy_filled, toy_source, toy_confidence = _fill_numeric_panel_arrays(
        toy_numeric, toy_mask_x, toy_lower, toy_upper,
        GapImputationConfig(max_gap=2),
    )
    toy_names, _ = numeric_feature_layout(
        toy_stable, toy_config, toy_numeric.shape[2]
    )

    first_batches = list(
        iter_causal_numeric_features(
            2,
            8,
            "train",
            stable_features=toy_stable,
            lower_guardrail=toy_lower,
            upper_guardrail=toy_upper,
            stock_cap=None,
            config=toy_config,
            numeric_values=toy_filled,
            fill_source_values=toy_source,
            fill_confidence_values=toy_confidence,
            observed_mask=toy_mask_x,
            label_mask=toy_mask_y,
            targets=toy_targets,
        )
    )
    original_outputs = [batch["features"].copy() for batch in first_batches]
    toy_numeric[8:] = 9999.0
    second_filled, second_source, second_confidence = _fill_numeric_panel_arrays(
        toy_numeric, toy_mask_x, toy_lower, toy_upper,
        GapImputationConfig(max_gap=2),
    )
    second_batches = list(
        iter_causal_numeric_features(
            2,
            8,
            "train",
            stable_features=toy_stable,
            lower_guardrail=toy_lower,
            upper_guardrail=toy_upper,
            stock_cap=None,
            config=toy_config,
            numeric_values=second_filled,
            fill_source_values=second_source,
            fill_confidence_values=second_confidence,
            observed_mask=toy_mask_x,
            label_mask=toy_mask_y,
            targets=toy_targets,
        )
    )

    assert len(first_batches) == len(second_batches) == 6
    for first, second, expected in zip(first_batches, second_batches, original_outputs):
        assert first["features"].shape[1] == len(toy_names)
        assert np.all(np.isfinite(first["features"]))
        assert np.allclose(first["features"], expected)
        assert np.allclose(first["features"], second["features"])
        rank_slice = slice(4, 8)
        assert np.max(np.abs(first["features"][:, rank_slice])) <= 1.0
        assert np.isclose(first["query_weights"].sum(), 1.0)
    gap_batch = next(batch for batch in first_batches if batch["time_idx"] == 4)
    assert np.array_equal(gap_batch["stocks"], np.arange(6, dtype=np.int32))
    assert gap_batch["features"].shape[0] == 6
    print("双重相对坐标数值特征自检通过。")


run_numeric_feature_self_test()

双重相对坐标数值特征自检通过。


### 7.6 第二部分的选择结果与下游接口

第二部分固定 `filled_num_x` 特征规则与指纹；后续缓存若与当前补全或特征配置不一致会直接拒绝复用。

In [21]:
SELECTED_FEATURE_CONFIG = {
    "numeric_config": NUMERIC_CONFIG,
    "stable_numeric_features": STABLE_NUMERIC_FEATURES.copy(),
    "feature_names": tuple(NUMERIC_FEATURE_NAMES),
    "feature_blocks": NUMERIC_FEATURE_BLOCKS.copy(),
    "feature_count": len(NUMERIC_FEATURE_NAMES),
    "numeric_feature_fingerprint": NUMERIC_FEATURE_FINGERPRINT,
    "numeric_input": "filled_num_x",
}

print(
    "第二部分已选择："
    f"{SELECTED_FEATURE_CONFIG['feature_count']} 个数值特征，"
    f"动态特征索引={STABLE_NUMERIC_FEATURES.tolist()}。"
)

第二部分已选择：363 个数值特征，动态特征索引=[90, 57, 40, 8, 82, 42, 11, 55, 48, 47, 91, 74, 39, 7, 68, 41, 56, 69, 33, 83]。


## 8. 第三部分：训练历史策略自动选择

本部分只在 Train 内比较训练历史的使用方式。每个候选共用相同特征、固定8轮 LambdaRank、全部合格股票和相同随机种子；官方 Valid 不参与选择。验证完成后直接设置 `SELECTED_TRAINING_CONFIG`，不写 Markdown 或 CSV 报告。

### 8.1 候选策略与走步切分

`recent_30` 保留为对用户提出的30期训练想法的直接检验；时间衰减策略保留全部历史，但降低旧时间点的总权重。

In [22]:
@dataclass(frozen=True)
class TrainingPolicy:
    name: str
    mode: str
    half_life: float | None = None
    lookback: int | None = None
    stock_cap: int | None = None
    equal_query_weight: bool = True

    def __post_init__(self) -> None:
        if self.mode not in {"full_equal", "exponential_decay", "recent_window"}:
            raise ValueError(f"未知训练策略模式：{self.mode}")
        if self.mode == "exponential_decay" and (
            self.half_life is None or self.half_life <= 0
        ):
            raise ValueError("时间衰减策略必须提供正数 half_life。")
        if self.mode == "recent_window" and (
            self.lookback is None or self.lookback <= 0
        ):
            raise ValueError("近期窗口策略必须提供正整数 lookback。")
        if self.stock_cap is not None:
            raise ValueError("第三部分正式候选不允许设置股票数量上限。")


@dataclass(frozen=True)
class WalkForwardFold:
    name: str
    train_start: int
    train_stop: int
    valid_start: int
    valid_stop: int


@dataclass(frozen=True)
class TrainingPolicySelectionConfig:
    fold_count: int = 3
    validation_length: int = 243
    label_bins: int = 64
    boost_rounds: int = 8
    worst_block_count: int = 4
    minimum_positive_folds: int = 2
    maximum_worst_block_drop: float = 0.003
    mean_weight: float = 0.50
    late_weight: float = 0.30
    worst_weight: float = 0.20
    fold_std_penalty: float = 0.10


TRAINING_SELECTION_CONFIG = TrainingPolicySelectionConfig()
TRAINING_POLICY_CANDIDATES = (
    TrainingPolicy("full_equal", "full_equal"),
    TrainingPolicy("decay_1200", "exponential_decay", half_life=1200.0),
    TrainingPolicy("decay_730", "exponential_decay", half_life=730.0),
    TrainingPolicy("recent_1702", "recent_window", lookback=1702),
    TrainingPolicy("recent_30", "recent_window", lookback=30),
)
POLICY_SCREEN_CACHE_DIR = NUMERIC_CACHE_DIR / "train_valid"
RUN_TRAINING_POLICY_SELECTION = True
REBUILD_TRAINING_POLICY_CACHE = False


def build_walk_forward_folds(
    train_start: int,
    train_stop: int,
    config: TrainingPolicySelectionConfig = TRAINING_SELECTION_CONFIG,
) -> list[WalkForwardFold]:
    first_valid_start = train_stop - config.fold_count * config.validation_length
    if first_valid_start <= train_start:
        raise ValueError("Train 长度不足以建立指定走步折。")
    folds = []
    for fold_idx in range(config.fold_count):
        valid_start = first_valid_start + fold_idx * config.validation_length
        valid_stop = valid_start + config.validation_length
        folds.append(
            WalkForwardFold(
                name=f"fold_{fold_idx + 1}",
                train_start=train_start,
                train_stop=valid_start,
                valid_start=valid_start,
                valid_stop=valid_stop,
            )
        )
    if folds[-1].valid_stop != train_stop:
        raise AssertionError("走步折没有覆盖到 Train 末端。")
    return folds


def policy_training_start(
    policy: TrainingPolicy,
    available_start: int,
    train_stop: int,
) -> int:
    if policy.mode == "recent_window":
        return max(available_start, train_stop - int(policy.lookback))
    return available_start


def policy_query_time_weights(
    policy: TrainingPolicy,
    training_times: np.ndarray,
    train_stop: int,
) -> np.ndarray:
    training_times = np.asarray(training_times, dtype=np.int64)
    if training_times.size == 0:
        raise ValueError("训练时间不能为空。")
    if policy.mode == "exponential_decay":
        age = (train_stop - 1 - training_times).astype(np.float64)
        weights = np.power(0.5, age / float(policy.half_life))
    else:
        weights = np.ones(training_times.size, dtype=np.float64)
    weights /= weights.mean()
    if not np.all(np.isfinite(weights)) or np.any(weights <= 0):
        raise ValueError("时间权重包含无效值。")
    return weights.astype(np.float32)


def combine_query_and_time_weights(
    base_query_weights: np.ndarray,
    group_sizes: np.ndarray,
    query_time_weights: np.ndarray,
) -> np.ndarray:
    group_sizes = np.asarray(group_sizes, dtype=np.int64)
    query_time_weights = np.asarray(query_time_weights, dtype=np.float64)
    if group_sizes.size != query_time_weights.size:
        raise ValueError("group_sizes 与时间权重长度不一致。")
    base_query_weights = np.asarray(base_query_weights, dtype=np.float64)
    if base_query_weights.size != int(group_sizes.sum()):
        raise ValueError("基础样本权重与分组行数不一致。")
    row_weights = base_query_weights * np.repeat(query_time_weights, group_sizes)
    row_weights *= row_weights.size / row_weights.sum()
    if not np.all(np.isfinite(row_weights)) or np.any(row_weights <= 0):
        raise ValueError("组合训练权重包含无效值。")
    return row_weights.astype(np.float32)


WALK_FORWARD_FOLDS = build_walk_forward_folds(TRAIN_START, VALID_START)
pd.DataFrame([fold.__dict__ for fold in WALK_FORWARD_FOLDS])

,name,train_start,train_stop,valid_start,valid_stop
0,fold_1,486,2189,2189,2432
1,fold_2,486,2432,2432,2675
2,fold_3,486,2675,2675,2918


### 8.2 复用第二部分缓存

策略比较只改变训练时间范围和权重，因此所有候选共用同一份全股票数值特征缓存。缓存属于可复用计算产物，不是实验报告。

In [23]:
@dataclass
class NumericTrainingCache:
    start: int
    stop: int
    features: np.ndarray
    labels: np.ndarray
    time_index: np.ndarray
    query_weights: np.ndarray
    group_sizes: np.ndarray
    group_offsets: np.ndarray
    feature_names: tuple[str, ...]
    role: str = "train"

    def time_slice(self, start: int, stop: int) -> slice:
        if not (self.start <= start < stop <= self.stop):
            raise ValueError(f"缓存时间范围外：[{start}, {stop})")
        left = int(self.group_offsets[start - self.start])
        right = int(self.group_offsets[stop - self.start])
        return slice(left, right)

    def groups_for(self, start: int, stop: int) -> np.ndarray:
        if not (self.start <= start < stop <= self.stop):
            raise ValueError(f"缓存时间范围外：[{start}, {stop})")
        return self.group_sizes[start - self.start : stop - self.start]


def numeric_cache_time_view(
    cache: NumericTrainingCache,
    start: int,
    stop: int,
    role: str,
) -> NumericTrainingCache:
    row_slice = cache.time_slice(start, stop)
    groups = np.asarray(cache.groups_for(start, stop), dtype=np.int64)
    offsets = np.concatenate(
        [np.array([0], dtype=np.int64), np.cumsum(groups, dtype=np.int64)]
    )
    return NumericTrainingCache(
        start=start, stop=stop, features=cache.features[row_slice],
        labels=cache.labels[row_slice], time_index=cache.time_index[row_slice],
        query_weights=cache.query_weights[row_slice], group_sizes=groups,
        group_offsets=offsets, feature_names=cache.feature_names, role=role,
    )


def open_numeric_training_cache(
    output_dir: Path,
    expected_role: str = "train",
) -> NumericTrainingCache:
    output_dir = Path(output_dir)
    metadata = json.loads((output_dir / "metadata.json").read_text(encoding="utf-8"))
    if metadata["role"] != expected_role:
        raise ValueError(
            f"缓存角色错误：{metadata['role']} != {expected_role}"
        )
    if tuple(metadata["feature_names"]) != tuple(NUMERIC_FEATURE_NAMES):
        raise ValueError("缓存特征名称与当前第二部分选择不一致。")
    if metadata.get("fingerprint") != NUMERIC_FEATURE_FINGERPRINT:
        raise ValueError("缓存指纹与当前补全及特征配置不一致。")

    features = np.load(output_dir / "features.npy", mmap_mode="r")
    labels = np.load(output_dir / "labels.npy", mmap_mode="r")
    time_index = np.load(output_dir / "time_index.npy", mmap_mode="r")
    query_weights = np.load(output_dir / "query_weights.npy", mmap_mode="r")
    group_sizes = np.load(output_dir / "group_sizes.npy").astype(np.int64)
    group_offsets = np.concatenate(
        [np.array([0], dtype=np.int64), np.cumsum(group_sizes, dtype=np.int64)]
    )
    row_count = int(group_offsets[-1])
    if features.shape != (row_count, len(NUMERIC_FEATURE_NAMES)):
        raise ValueError(f"缓存特征形状错误：{features.shape}")
    if labels.shape != (row_count,) or time_index.shape != (row_count,):
        raise ValueError("缓存标签或时间索引形状错误。")
    if query_weights.shape != (row_count,):
        raise ValueError("缓存样本权重形状错误。")
    if not np.all(np.isfinite(labels)) or not np.all(np.isfinite(query_weights)):
        raise ValueError("缓存标签或权重包含非有限值。")

    start = int(metadata["start"])
    stop = int(metadata["stop"])
    if group_sizes.size != stop - start:
        raise ValueError("缓存分组数量与时间区间不一致。")
    nonempty = np.flatnonzero(group_sizes > 0)
    if nonempty.size:
        first_rows = group_offsets[nonempty]
        expected_times = start + nonempty
        if not np.array_equal(time_index[first_rows], expected_times):
            raise ValueError("缓存时间索引与分组边界不一致。")

    return NumericTrainingCache(
        start=start,
        stop=stop,
        features=features,
        labels=labels,
        time_index=time_index,
        query_weights=query_weights,
        group_sizes=group_sizes,
        group_offsets=group_offsets,
        feature_names=tuple(metadata["feature_names"]),
        role=str(metadata["role"]),
    )


def ensure_numeric_training_cache(
    start: int = TRAIN_START,
    stop: int = TEST_START,
    output_dir: Path = POLICY_SCREEN_CACHE_DIR,
    rebuild: bool = REBUILD_TRAINING_POLICY_CACHE,
) -> NumericTrainingCache:
    output_dir = Path(output_dir)
    required = [
        output_dir / name
        for name in (
            "features.npy", "labels.npy", "time_index.npy",
            "query_weights.npy", "group_sizes.npy", "metadata.json",
        )
    ]
    complete = all(path.exists() for path in required)
    if complete and not rebuild:
        cache = open_numeric_training_cache(output_dir)
        if cache.start != start or cache.stop != stop:
            raise ValueError("现有缓存时间范围不匹配；请显式启用 rebuild。")
        return cache
    if not complete and any(path.exists() for path in required) and not rebuild:
        raise FileExistsError("发现不完整缓存；确认后设置 REBUILD_TRAINING_POLICY_CACHE=True。")

    materialize_numeric_feature_cache(
        start,
        stop,
        "train",
        output_dir,
        stock_cap=None,
        overwrite=rebuild,
    )
    return open_numeric_training_cache(output_dir)

### 8.3 固定筛选模型与 RankIC 计算

连续 `y1` 在每个时间截面独立转换为64档相关度。筛选阶段固定模型参数，不进行调参；不同候选之间只允许训练历史与时间权重不同。

In [24]:
def relevance_by_query(
    labels: np.ndarray,
    group_sizes: np.ndarray,
    bins: int,
) -> np.ndarray:
    if bins < 2 or bins > 255:
        raise ValueError("bins 必须位于 [2, 255]。")
    labels = np.asarray(labels)
    group_sizes = np.asarray(group_sizes, dtype=np.int64)
    if labels.size != int(group_sizes.sum()):
        raise ValueError("标签长度与分组大小不一致。")
    relevance = np.zeros(labels.size, dtype=np.uint8)
    offset = 0
    for group_size in group_sizes:
        group_size = int(group_size)
        next_offset = offset + group_size
        if group_size > 1:
            ranks = rankdata(labels[offset:next_offset], method="average")
            relevance[offset:next_offset] = np.rint(
                (ranks - 1.0) / (group_size - 1.0) * (bins - 1)
            ).astype(np.uint8)
        offset = next_offset
    return relevance


def per_query_rank_ic(
    predictions: np.ndarray,
    labels: np.ndarray,
    group_sizes: np.ndarray,
) -> np.ndarray:
    predictions = np.asarray(predictions, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    group_sizes = np.asarray(group_sizes, dtype=np.int64)
    if predictions.shape != labels.shape or labels.size != int(group_sizes.sum()):
        raise ValueError("预测、标签和分组大小不一致。")
    values = np.full(group_sizes.size, np.nan, dtype=np.float64)
    offset = 0
    for group_idx, group_size in enumerate(group_sizes):
        group_size = int(group_size)
        next_offset = offset + group_size
        if group_size > 1:
            prediction_rank = rankdata(predictions[offset:next_offset], method="average")
            label_rank = rankdata(labels[offset:next_offset], method="average")
            if np.std(prediction_rank) > 0 and np.std(label_rank) > 0:
                values[group_idx] = np.corrcoef(prediction_rank, label_rank)[0, 1]
        offset = next_offset
    return values


def summarize_rank_ic(
    rank_ic_values: np.ndarray,
    block_count: int,
) -> dict[str, float]:
    rank_ic_values = np.asarray(rank_ic_values, dtype=np.float64)
    finite = np.isfinite(rank_ic_values)
    if not np.any(finite):
        raise ValueError("没有可用的逐时点 RankIC。")
    usable = rank_ic_values[finite]
    late_start = usable.size // 2
    blocks = [
        block for block in np.array_split(usable, min(block_count, usable.size))
        if block.size
    ]
    return {
        "time_points": int(usable.size),
        "mean_rank_ic": float(np.mean(usable)),
        "late_rank_ic": float(np.mean(usable[late_start:])),
        "worst_block_rank_ic": float(min(np.mean(block) for block in blocks)),
    }


def fixed_policy_screen_parameters(
    bins: int,
    seed: int,
) -> dict:
    return {
        "objective": "lambdarank",
        "metric": "None",
        "learning_rate": 0.0228695,
        "num_leaves": 79,
        "min_data_in_leaf": 147,
        "feature_fraction": 0.80936,
        "bagging_fraction": 0.647764,
        "bagging_freq": 1,
        "lambda_l1": 2.35724,
        "lambda_l2": 0.238705,
        "max_bin": 127,
        "label_gain": list(range(bins)),
        "lambdarank_truncation_level": 1024,
        "lambdarank_norm": True,
        "deterministic": True,
        "force_col_wise": True,
        "seed": seed,
        "feature_fraction_seed": seed,
        "bagging_seed": seed,
        "verbosity": -1,
    }


def evaluate_policy_on_fold(
    cache: NumericTrainingCache,
    all_relevance: np.ndarray,
    fold: WalkForwardFold,
    policy: TrainingPolicy,
    lgb_module,
    config: TrainingPolicySelectionConfig = TRAINING_SELECTION_CONFIG,
    seed: int = SEED,
) -> dict:
    effective_start = policy_training_start(policy, fold.train_start, fold.train_stop)
    train_slice = cache.time_slice(effective_start, fold.train_stop)
    valid_slice = cache.time_slice(fold.valid_start, fold.valid_stop)
    train_groups = cache.groups_for(effective_start, fold.train_stop)
    valid_groups = cache.groups_for(fold.valid_start, fold.valid_stop)
    training_times = np.arange(effective_start, fold.train_stop, dtype=np.int64)
    time_weights = policy_query_time_weights(policy, training_times, fold.train_stop)
    sample_weights = combine_query_and_time_weights(
        cache.query_weights[train_slice], train_groups, time_weights
    )

    train_set = lgb_module.Dataset(
        cache.features[train_slice],
        label=all_relevance[train_slice],
        group=train_groups,
        weight=sample_weights,
        feature_name=list(cache.feature_names),
        free_raw_data=True,
    )
    model = lgb_module.train(
        fixed_policy_screen_parameters(config.label_bins, seed),
        train_set,
        num_boost_round=config.boost_rounds,
    )
    predictions = model.predict(
        cache.features[valid_slice],
        num_iteration=config.boost_rounds,
    )
    rank_ic_values = per_query_rank_ic(
        predictions, cache.labels[valid_slice], valid_groups
    )
    metrics = summarize_rank_ic(rank_ic_values, config.worst_block_count)
    result = {
        "fold": fold.name,
        "policy": policy.name,
        "effective_train_start": effective_start,
        "train_stop": fold.train_stop,
        "valid_start": fold.valid_start,
        "valid_stop": fold.valid_stop,
        "train_time_points": int(train_groups.size),
        "train_rows": int(train_groups.sum()),
        **metrics,
    }
    del train_set, model, predictions, rank_ic_values, sample_weights
    gc.collect()
    return result

### 8.4 自动选择与下游训练视图

候选必须至少在2个走步折优于全历史等权基准、平均值更高，且最差区间的退化不超过0.003；否则自动回退到全历史等权。

In [25]:
def summarize_training_policy_results(
    fold_results: pd.DataFrame,
    policies: tuple[TrainingPolicy, ...] = TRAINING_POLICY_CANDIDATES,
    config: TrainingPolicySelectionConfig = TRAINING_SELECTION_CONFIG,
) -> tuple[pd.DataFrame, TrainingPolicy]:
    required_columns = {
        "fold", "policy", "mean_rank_ic",
        "late_rank_ic", "worst_block_rank_ic",
    }
    if not required_columns.issubset(fold_results.columns):
        raise ValueError("折结果缺少自动选择所需列。")
    policy_by_name = {policy.name: policy for policy in policies}
    if "full_equal" not in policy_by_name:
        raise ValueError("候选中必须包含 full_equal 基准。")
    if fold_results.duplicated(["fold", "policy"]).any():
        raise ValueError("同一折与策略出现重复结果。")

    baseline_by_fold = (
        fold_results[fold_results["policy"] == "full_equal"]
        .set_index("fold")["mean_rank_ic"]
    )
    expected_folds = set(fold_results["fold"])
    if set(baseline_by_fold.index) != expected_folds:
        raise ValueError("每个走步折都必须包含 full_equal 基准。")

    rows = []
    for policy_name, group in fold_results.groupby("policy", sort=False):
        if policy_name not in policy_by_name:
            raise ValueError(f"结果包含未知策略：{policy_name}")
        aligned_baseline = group["fold"].map(baseline_by_fold)
        positive_fold_count = int(
            np.count_nonzero(group["mean_rank_ic"].to_numpy() > aligned_baseline.to_numpy())
        )
        rows.append(
            {
                "policy": policy_name,
                "fold_count": int(group.shape[0]),
                "mean_rank_ic": float(group["mean_rank_ic"].mean()),
                "late_rank_ic": float(group["late_rank_ic"].mean()),
                "worst_block_rank_ic": float(group["worst_block_rank_ic"].min()),
                "fold_std": float(group["mean_rank_ic"].std(ddof=0)),
                "positive_fold_count": positive_fold_count,
            }
        )
    summary = pd.DataFrame(rows)
    baseline = summary.loc[summary["policy"] == "full_equal"].iloc[0]
    summary["mean_improvement"] = summary["mean_rank_ic"] - baseline["mean_rank_ic"]
    summary["selection_score"] = (
        config.mean_weight * summary["mean_rank_ic"]
        + config.late_weight * summary["late_rank_ic"]
        + config.worst_weight * summary["worst_block_rank_ic"]
        - config.fold_std_penalty * summary["fold_std"]
    )
    summary["qualified"] = (
        (summary["mean_improvement"] > 0.0)
        & (summary["positive_fold_count"] >= config.minimum_positive_folds)
        & (
            summary["worst_block_rank_ic"]
            >= baseline["worst_block_rank_ic"] - config.maximum_worst_block_drop
        )
    )
    summary.loc[summary["policy"] == "full_equal", "qualified"] = True
    selected_row = summary[summary["qualified"]].sort_values(
        ["selection_score", "mean_rank_ic", "policy"],
        ascending=[False, False, True],
        kind="stable",
    ).iloc[0]
    selected_policy = policy_by_name[str(selected_row["policy"])]
    summary["selected"] = summary["policy"] == selected_policy.name
    summary = summary.sort_values(
        ["selected", "selection_score"], ascending=[False, False], kind="stable"
    ).reset_index(drop=True)
    return summary, selected_policy


def run_training_policy_selection(
    cache: NumericTrainingCache,
    policies: tuple[TrainingPolicy, ...] = TRAINING_POLICY_CANDIDATES,
    folds: list[WalkForwardFold] = WALK_FORWARD_FOLDS,
    config: TrainingPolicySelectionConfig = TRAINING_SELECTION_CONFIG,
    lgb_module=None,
) -> tuple[pd.DataFrame, pd.DataFrame, TrainingPolicy]:
    if lgb_module is None:
        try:
            import lightgbm as lgb_module
        except ImportError as error:
            raise ImportError("请使用 jingge_ts Anaconda 环境运行策略筛选。") from error

    if cache.start > min(fold.train_start for fold in folds):
        raise ValueError("缓存没有覆盖最早走步训练起点。")
    if cache.stop < max(fold.valid_stop for fold in folds):
        raise ValueError("缓存没有覆盖全部走步验证区间。")
    all_relevance = relevance_by_query(
        cache.labels, cache.group_sizes, config.label_bins
    )
    rows = []
    for fold_idx, fold in enumerate(folds):
        for policy in policies:
            print(f"验证 {fold.name} / {policy.name} ...")
            rows.append(
                evaluate_policy_on_fold(
                    cache,
                    all_relevance,
                    fold,
                    policy,
                    lgb_module,
                    config,
                    seed=SEED + fold_idx * 101,
                )
            )
    fold_results = pd.DataFrame(rows)
    summary, selected_policy = summarize_training_policy_results(
        fold_results, policies, config
    )
    return fold_results, summary, selected_policy


def build_selected_training_view(
    cache: NumericTrainingCache,
    training_stop: int,
    policy: TrainingPolicy | None = None,
    label_bins: int = TRAINING_SELECTION_CONFIG.label_bins,
) -> dict:
    policy = SELECTED_TRAINING_CONFIG if policy is None else policy
    effective_start = policy_training_start(policy, cache.start, training_stop)
    row_slice = cache.time_slice(effective_start, training_stop)
    groups = cache.groups_for(effective_start, training_stop)
    times = np.arange(effective_start, training_stop, dtype=np.int64)
    time_weights = policy_query_time_weights(policy, times, training_stop)
    sample_weights = combine_query_and_time_weights(
        cache.query_weights[row_slice], groups, time_weights
    )
    relevance = relevance_by_query(cache.labels[row_slice], groups, label_bins)
    return {
        "policy": policy,
        "start": effective_start,
        "stop": training_stop,
        "features": cache.features[row_slice],
        "labels": cache.labels[row_slice],
        "relevance": relevance,
        "groups": groups,
        "sample_weights": sample_weights,
        "feature_names": cache.feature_names,
    }

### 8.5 选择逻辑自检

合成测试检查走步折边界、每时点等总权重、时间衰减方向、30期窗口，以及自动选择和回退规则。

In [26]:
def run_training_policy_self_test() -> None:
    toy_config = TrainingPolicySelectionConfig(
        fold_count=3, validation_length=10, minimum_positive_folds=2
    )
    toy_folds = build_walk_forward_folds(0, 100, toy_config)
    assert [(fold.valid_start, fold.valid_stop) for fold in toy_folds] == [
        (70, 80), (80, 90), (90, 100)
    ]

    decay_policy = TrainingPolicy("toy_decay", "exponential_decay", half_life=10.0)
    decay_weights = policy_query_time_weights(decay_policy, np.arange(20), 20)
    assert decay_weights[-1] > decay_weights[0]
    assert np.isclose(decay_weights.mean(), 1.0)
    recent_policy = TrainingPolicy("toy_recent", "recent_window", lookback=30)
    assert policy_training_start(recent_policy, 0, 100) == 70

    group_sizes = np.array([2, 3, 1], dtype=np.int32)
    base_weights = np.concatenate(
        [np.full(size, 1.0 / size) for size in group_sizes]
    )
    combined = combine_query_and_time_weights(
        base_weights, group_sizes, np.ones(3, dtype=np.float32)
    )
    offsets = np.concatenate([[0], np.cumsum(group_sizes)])
    group_weight_sums = np.array(
        [combined[offsets[idx] : offsets[idx + 1]].sum() for idx in range(3)]
    )
    assert np.allclose(group_weight_sums, group_weight_sums[0])

    toy_rows = []
    for fold_idx in range(3):
        toy_rows.extend(
            [
                {"fold": f"fold_{fold_idx + 1}", "policy": "full_equal",
                 "mean_rank_ic": 0.10, "late_rank_ic": 0.09,
                 "worst_block_rank_ic": 0.06},
                {"fold": f"fold_{fold_idx + 1}", "policy": "decay_730",
                 "mean_rank_ic": 0.11, "late_rank_ic": 0.105,
                 "worst_block_rank_ic": 0.061},
            ]
        )
    toy_policies = (
        TrainingPolicy("full_equal", "full_equal"),
        TrainingPolicy("decay_730", "exponential_decay", half_life=730.0),
    )
    _, selected = summarize_training_policy_results(
        pd.DataFrame(toy_rows), toy_policies, toy_config
    )
    assert selected.name == "decay_730"
    print("训练历史策略选择自检通过。")


run_training_policy_self_test()

训练历史策略选择自检通过。


### 8.6 执行验证并确定后续配置

首次执行会构建全量 Train 数值特征缓存，耗时与磁盘占用较大。完成后只显示候选表和选中的配置，不写实验报告。

In [27]:
training_policy_fold_results = None
training_policy_summary = None
training_policy_cache = None

if RUN_TRAINING_POLICY_SELECTION:
    training_policy_cache = ensure_numeric_training_cache()
    (
        training_policy_fold_results,
        training_policy_summary,
        SELECTED_TRAINING_CONFIG,
    ) = run_training_policy_selection(training_policy_cache)
else:
    SELECTED_TRAINING_CONFIG = TRAINING_POLICY_CANDIDATES[0]
    print("训练历史策略验证未执行，暂时回退到 full_equal。")

ACTIVE_PIPELINE_CONFIG = {
    "sample": SELECTED_SAMPLE_CONFIG,
    "feature": SELECTED_FEATURE_CONFIG,
    "training": SELECTED_TRAINING_CONFIG,
}

if training_policy_summary is not None:
    display(
        training_policy_summary[
            [
                "policy", "mean_rank_ic", "late_rank_ic",
                "worst_block_rank_ic", "fold_std",
                "positive_fold_count", "selection_score",
                "qualified", "selected",
            ]
        ]
    )
    selected_row = training_policy_summary[training_policy_summary["selected"]].iloc[0]
    print("\n训练时间策略验证完成")
    print(f"自动选择：{SELECTED_TRAINING_CONFIG.name}")
    print(f"走步平均 RankIC：{selected_row['mean_rank_ic']:.6f}")
    print(f"后段 RankIC：{selected_row['late_rank_ic']:.6f}")
    print(f"最差区间 RankIC：{selected_row['worst_block_rank_ic']:.6f}")
    print(
        "优于全历史折数："
        f"{int(selected_row['positive_fold_count'])} / {TRAINING_SELECTION_CONFIG.fold_count}"
    )

pd.Series(
    {
        "selected_policy": SELECTED_TRAINING_CONFIG.name,
        "mode": SELECTED_TRAINING_CONFIG.mode,
        "half_life": SELECTED_TRAINING_CONFIG.half_life,
        "lookback": SELECTED_TRAINING_CONFIG.lookback,
        "stock_cap": SELECTED_TRAINING_CONFIG.stock_cap,
        "equal_query_weight": SELECTED_TRAINING_CONFIG.equal_query_weight,
    },
    name="active_training_config",
)

验证 fold_1 / full_equal ...
验证 fold_1 / decay_1200 ...
验证 fold_1 / decay_730 ...
验证 fold_1 / recent_1702 ...
验证 fold_1 / recent_30 ...
验证 fold_2 / full_equal ...
验证 fold_2 / decay_1200 ...
验证 fold_2 / decay_730 ...
验证 fold_2 / recent_1702 ...
验证 fold_2 / recent_30 ...
验证 fold_3 / full_equal ...
验证 fold_3 / decay_1200 ...
验证 fold_3 / decay_730 ...
验证 fold_3 / recent_1702 ...
验证 fold_3 / recent_30 ...


,policy,mean_rank_ic,late_rank_ic,worst_block_rank_ic,fold_std,positive_fold_count,selection_score,qualified,selected
0,decay_1200,0.094782,0.092184,0.052174,0.011038,3,0.084377,True,True
1,decay_730,0.096429,0.092217,0.042389,0.014209,3,0.082936,True,False
2,full_equal,0.092739,0.089753,0.038793,0.013141,0,0.079740,True,False
3,recent_1702,0.091569,0.089544,0.034857,0.013792,1,0.078240,False,False
4,recent_30,0.051058,0.046014,-0.007851,0.023539,0,0.035409,False,False



训练时间策略验证完成
自动选择：decay_1200
走步平均 RankIC：0.094782
后段 RankIC：0.092184
最差区间 RankIC：0.052174
优于全历史折数：3 / 3


selected_policy              decay_1200
mode                  exponential_decay
half_life                        1200.0
lookback                           None
stock_cap                          None
equal_query_weight                 True
Name: active_training_config, dtype: object

## 9. 第四部分：模型验证与手动选择

本部分固定前三部分的样本、特征和训练时间策略，只比较排序目标与训练轮数。验证结果不会自动覆盖用户选择；真正使用的模型由手动选择单元格中唯一未被 `#` 注释的名称决定。

### 9.1 模型候选与执行开关

8轮 LambdaRank 仅作为旧管道的历史参考；新补全、特征与 mask_y 样本口径改变后，不能把旧线上 `0.108105` 当作可直接复现的成绩。其余候选只改变轮数或排序目标。

In [28]:
@dataclass(frozen=True)
class ModelCandidate:
    name: str
    objective: str
    boost_rounds: int
    note: str

    def __post_init__(self) -> None:
        if self.objective not in {"lambdarank", "rank_xendcg"}:
            raise ValueError(f"未知排序目标：{self.objective}")
        if self.boost_rounds <= 0:
            raise ValueError("boost_rounds 必须为正整数。")


MODEL_CANDIDATES = (
    ModelCandidate(
        "lambdarank_8", "lambdarank", 8,
        "旧管道 0.108105 的参考配置；新管道需重新验证",
    ),
    ModelCandidate(
        "lambdarank_16", "lambdarank", 16,
        "容量稍高，检验新特征是否需要更多轮数",
    ),
    ModelCandidate(
        "lambdarank_32", "lambdarank", 32,
        "更强拟合能力，同时有更高过拟合风险",
    ),
    ModelCandidate(
        "xendcg_16", "rank_xendcg", 16,
        "不同排序目标的对照候选",
    ),
)
MODEL_CANDIDATE_BY_NAME = {candidate.name: candidate for candidate in MODEL_CANDIDATES}
MODEL_BASELINE_NAME = "lambdarank_16"
MODEL_VALID_CACHE_DIR = NUMERIC_CACHE_DIR / "valid"
RUN_MODEL_CANDIDATE_VALIDATION = True
RUN_OFFICIAL_VALID_CHECK = True
REBUILD_MODEL_VALID_CACHE = False
OFFICIAL_VALID_WARNING_DROP = 0.001

pd.DataFrame(
    [
        {
            "name": candidate.name,
            "objective": candidate.objective,
            "boost_rounds": candidate.boost_rounds,
            "note": candidate.note,
        }
        for candidate in MODEL_CANDIDATES
    ]
)

,name,objective,boost_rounds,note
0,lambdarank_8,lambdarank,8,旧管道 0.108105 的参考配置；新管道需重新验证
1,lambdarank_16,lambdarank,16,容量稍高，检验新特征是否需要更多轮数
2,lambdarank_32,lambdarank,32,更强拟合能力，同时有更高过拟合风险
3,xendcg_16,rank_xendcg,16,不同排序目标的对照候选


### 9.2 固定数据口径比较候选

每个候选使用第三部分选出的同一时间策略、同一走步折、全部股票、同一特征和同一随机种子。

In [29]:
def model_candidate_parameters(
    candidate: ModelCandidate,
    bins: int,
    seed: int,
) -> dict:
    parameters = fixed_policy_screen_parameters(bins, seed)
    parameters["objective"] = candidate.objective
    if candidate.objective == "rank_xendcg":
        parameters.pop("label_gain", None)
        parameters.pop("lambdarank_truncation_level", None)
        parameters.pop("lambdarank_norm", None)
        parameters["objective_seed"] = seed
    return parameters


def evaluate_model_candidate_on_fold(
    cache: NumericTrainingCache,
    all_relevance: np.ndarray,
    fold: WalkForwardFold,
    training_policy: TrainingPolicy,
    candidate: ModelCandidate,
    lgb_module,
    config: TrainingPolicySelectionConfig = TRAINING_SELECTION_CONFIG,
    seed: int = SEED,
) -> dict:
    effective_start = policy_training_start(
        training_policy, fold.train_start, fold.train_stop
    )
    train_slice = cache.time_slice(effective_start, fold.train_stop)
    valid_slice = cache.time_slice(fold.valid_start, fold.valid_stop)
    train_groups = cache.groups_for(effective_start, fold.train_stop)
    valid_groups = cache.groups_for(fold.valid_start, fold.valid_stop)
    training_times = np.arange(effective_start, fold.train_stop, dtype=np.int64)
    time_weights = policy_query_time_weights(
        training_policy, training_times, fold.train_stop
    )
    sample_weights = combine_query_and_time_weights(
        cache.query_weights[train_slice], train_groups, time_weights
    )

    train_set = lgb_module.Dataset(
        cache.features[train_slice],
        label=all_relevance[train_slice],
        group=train_groups,
        weight=sample_weights,
        feature_name=list(cache.feature_names),
        free_raw_data=True,
    )
    model = lgb_module.train(
        model_candidate_parameters(candidate, config.label_bins, seed),
        train_set,
        num_boost_round=candidate.boost_rounds,
    )
    predictions = model.predict(
        cache.features[valid_slice],
        num_iteration=candidate.boost_rounds,
    )
    rank_ic_values = per_query_rank_ic(
        predictions, cache.labels[valid_slice], valid_groups
    )
    metrics = summarize_rank_ic(rank_ic_values, config.worst_block_count)
    result = {
        "fold": fold.name,
        "model": candidate.name,
        "objective": candidate.objective,
        "boost_rounds": candidate.boost_rounds,
        "effective_train_start": effective_start,
        "train_stop": fold.train_stop,
        "valid_start": fold.valid_start,
        "valid_stop": fold.valid_stop,
        "train_time_points": int(train_groups.size),
        "train_rows": int(train_groups.sum()),
        **metrics,
    }
    del train_set, model, predictions, rank_ic_values, sample_weights
    gc.collect()
    return result


def summarize_model_candidate_results(
    fold_results: pd.DataFrame,
    candidates: tuple[ModelCandidate, ...] = MODEL_CANDIDATES,
    config: TrainingPolicySelectionConfig = TRAINING_SELECTION_CONFIG,
) -> pd.DataFrame:
    required_columns = {
        "fold", "model", "mean_rank_ic",
        "late_rank_ic", "worst_block_rank_ic",
    }
    if not required_columns.issubset(fold_results.columns):
        raise ValueError("模型折结果缺少汇总所需列。")
    if fold_results.duplicated(["fold", "model"]).any():
        raise ValueError("同一折与模型出现重复结果。")
    candidate_names = {candidate.name for candidate in candidates}
    if set(fold_results["model"]) - candidate_names:
        raise ValueError("模型折结果包含未知候选。")
    baseline_by_fold = (
        fold_results[fold_results["model"] == MODEL_BASELINE_NAME]
        .set_index("fold")["mean_rank_ic"]
    )
    expected_folds = set(fold_results["fold"])
    if set(baseline_by_fold.index) != expected_folds:
        raise ValueError("每个走步折都必须包含8轮 LambdaRank 基准。")

    rows = []
    for model_name, group in fold_results.groupby("model", sort=False):
        aligned_baseline = group["fold"].map(baseline_by_fold)
        rows.append(
            {
                "model": model_name,
                "fold_count": int(group.shape[0]),
                "mean_rank_ic": float(group["mean_rank_ic"].mean()),
                "late_rank_ic": float(group["late_rank_ic"].mean()),
                "worst_block_rank_ic": float(group["worst_block_rank_ic"].min()),
                "fold_std": float(group["mean_rank_ic"].std(ddof=0)),
                "positive_fold_count": int(
                    np.count_nonzero(
                        group["mean_rank_ic"].to_numpy() > aligned_baseline.to_numpy()
                    )
                ),
            }
        )
    summary = pd.DataFrame(rows)
    baseline = summary.loc[summary["model"] == MODEL_BASELINE_NAME].iloc[0]
    summary["mean_improvement"] = summary["mean_rank_ic"] - baseline["mean_rank_ic"]
    summary["validation_score"] = (
        config.mean_weight * summary["mean_rank_ic"]
        + config.late_weight * summary["late_rank_ic"]
        + config.worst_weight * summary["worst_block_rank_ic"]
        - config.fold_std_penalty * summary["fold_std"]
    )
    summary["passes_anchor_gate"] = (
        (summary["mean_improvement"] > 0.0)
        & (summary["positive_fold_count"] >= config.minimum_positive_folds)
        & (
            summary["worst_block_rank_ic"]
            >= baseline["worst_block_rank_ic"] - config.maximum_worst_block_drop
        )
    )
    summary.loc[summary["model"] == MODEL_BASELINE_NAME, "passes_anchor_gate"] = True
    qualified = summary[summary["passes_anchor_gate"]]
    recommended_name = str(
        qualified.sort_values(
            ["validation_score", "mean_rank_ic", "model"],
            ascending=[False, False, True],
            kind="stable",
        ).iloc[0]["model"]
    )
    summary["validation_recommendation"] = summary["model"] == recommended_name
    return summary.sort_values(
        ["validation_recommendation", "validation_score"],
        ascending=[False, False],
        kind="stable",
    ).reset_index(drop=True)


def run_model_candidate_validation(
    cache: NumericTrainingCache,
    training_policy: TrainingPolicy,
    candidates: tuple[ModelCandidate, ...] = MODEL_CANDIDATES,
    folds: list[WalkForwardFold] = WALK_FORWARD_FOLDS,
    config: TrainingPolicySelectionConfig = TRAINING_SELECTION_CONFIG,
    lgb_module=None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if lgb_module is None:
        try:
            import lightgbm as lgb_module
        except ImportError as error:
            raise ImportError("请使用 jingge_ts Anaconda 环境运行模型验证。") from error
    all_relevance = relevance_by_query(
        cache.labels, cache.group_sizes, config.label_bins
    )
    rows = []
    for fold_idx, fold in enumerate(folds):
        for candidate in candidates:
            print(f"验证模型 {fold.name} / {candidate.name} ...")
            rows.append(
                evaluate_model_candidate_on_fold(
                    cache, all_relevance, fold, training_policy, candidate,
                    lgb_module, config, seed=SEED + fold_idx * 101,
                )
            )
    fold_results = pd.DataFrame(rows)
    return fold_results, summarize_model_candidate_results(
        fold_results, candidates, config
    )

### 9.3 选择接口自检

自检确保必须且只能保留一个未注释模型，并确认验证建议不会替代手动选择。

In [30]:
def resolve_manual_model_selection(
    selected_names: list[str],
    candidate_by_name: dict[str, ModelCandidate] = MODEL_CANDIDATE_BY_NAME,
) -> ModelCandidate:
    if len(selected_names) != 1:
        raise ValueError("MANUAL_MODEL_SELECTION 必须且只能保留一个未注释模型。")
    selected_name = selected_names[0]
    if selected_name not in candidate_by_name:
        raise ValueError(f"未知手动模型：{selected_name}")
    return candidate_by_name[selected_name]


def run_model_selection_self_test() -> None:
    assert resolve_manual_model_selection(["lambdarank_8"]).name == "lambdarank_8"
    for invalid in ([], ["lambdarank_8", "lambdarank_16"]):
        try:
            resolve_manual_model_selection(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError("手动模型数量检查未生效。")

    toy_rows = []
    for fold_idx in range(3):
        toy_rows.extend(
            [
                {"fold": f"fold_{fold_idx + 1}", "model": "lambdarank_8",
                 "mean_rank_ic": 0.10, "late_rank_ic": 0.09,
                 "worst_block_rank_ic": 0.06},
                {"fold": f"fold_{fold_idx + 1}", "model": "lambdarank_16",
                 "mean_rank_ic": 0.11, "late_rank_ic": 0.105,
                 "worst_block_rank_ic": 0.061},
            ]
        )
    toy_candidates = (
        MODEL_CANDIDATE_BY_NAME["lambdarank_8"],
        MODEL_CANDIDATE_BY_NAME["lambdarank_16"],
    )
    toy_summary = summarize_model_candidate_results(
        pd.DataFrame(toy_rows), toy_candidates
    )
    assert toy_summary.loc[toy_summary["validation_recommendation"], "model"].iloc[0] == (
        "lambdarank_16"
    )
    assert resolve_manual_model_selection(["lambdarank_8"]).name == "lambdarank_8"
    print("模型验证与手动选择接口自检通过。")


run_model_selection_self_test()

模型验证与手动选择接口自检通过。


### 9.4 执行 Train 内模型验证

此表只提供建议。无论推荐列显示什么，都不会改变下一单元格中的手动选择。

In [31]:
model_candidate_fold_results = None
model_candidate_summary = None
needs_model_training_cache = (
    RUN_MODEL_CANDIDATE_VALIDATION or RUN_OFFICIAL_VALID_CHECK
)
model_training_cache = training_policy_cache
if model_training_cache is None and needs_model_training_cache:
    model_training_cache = ensure_numeric_training_cache()

if RUN_MODEL_CANDIDATE_VALIDATION:
    model_candidate_fold_results, model_candidate_summary = (
        run_model_candidate_validation(
            model_training_cache, ACTIVE_PIPELINE_CONFIG["training"]
        )
    )
    display(
        model_candidate_summary[
            [
                "model", "mean_rank_ic", "late_rank_ic",
                "worst_block_rank_ic", "fold_std",
                "positive_fold_count", "mean_improvement",
                "validation_score", "passes_anchor_gate",
                "validation_recommendation",
            ]
        ]
    )
    validation_recommended_model = str(
        model_candidate_summary.loc[
            model_candidate_summary["validation_recommendation"], "model"
        ].iloc[0]
    )
    print(f"验证建议：{validation_recommended_model}；最终仍以手动选择为准。")
else:
    validation_recommended_model = None
    print("Train 内模型验证未执行；仍可手动选择模型。")

验证模型 fold_1 / lambdarank_8 ...
验证模型 fold_1 / lambdarank_16 ...
验证模型 fold_1 / lambdarank_32 ...
验证模型 fold_1 / xendcg_16 ...
验证模型 fold_2 / lambdarank_8 ...
验证模型 fold_2 / lambdarank_16 ...
验证模型 fold_2 / lambdarank_32 ...
验证模型 fold_2 / xendcg_16 ...
验证模型 fold_3 / lambdarank_8 ...
验证模型 fold_3 / lambdarank_16 ...
验证模型 fold_3 / lambdarank_32 ...
验证模型 fold_3 / xendcg_16 ...


,model,mean_rank_ic,late_rank_ic,worst_block_rank_ic,fold_std,positive_fold_count,mean_improvement,validation_score,passes_anchor_gate,validation_recommendation
0,lambdarank_32,0.100747,0.096624,0.054222,0.013405,3,0.003741,0.088864,True,True
1,lambdarank_16,0.097006,0.093623,0.051925,0.012342,0,0.000000,0.085741,True,False
2,lambdarank_8,0.094782,0.092184,0.052174,0.011038,0,-0.002224,0.084377,False,False
3,xendcg_16,0.069425,0.078604,0.011175,0.010400,0,-0.027582,0.059489,False,False


验证建议：lambdarank_32；最终仍以手动选择为准。


### 9.5 手动选择模型并填写备注

请保证列表中只有一行没有 `#`。如需切换模型，给当前行加 `#`，再删除目标行前的 `#`；验证代码不会修改这里。

In [32]:
MANUAL_MODEL_SELECTION = [
    #"lambdarank_8",       # 旧管道参考配置；新补全口径下需重新验证
    #"lambdarank_16",    # 容量稍高，观察新特征是否需要更多轮数
    "lambdarank_32",    # 拟合更强，过拟合风险也更高
    # "xendcg_16",        # 不同排序目标，用作对照
]
MODEL_SELECTION_NOTE = "默认选择旧管道参考的16轮 LambdaRank，新管道结果以当前验证为准。"

SELECTED_MODEL_CONFIG = resolve_manual_model_selection(MANUAL_MODEL_SELECTION)
ACTIVE_PIPELINE_CONFIG["model"] = SELECTED_MODEL_CONFIG
ACTIVE_PIPELINE_CONFIG["model_selection_note"] = MODEL_SELECTION_NOTE

print(f"手动选择模型：{SELECTED_MODEL_CONFIG.name}")
print(f"选择备注：{MODEL_SELECTION_NOTE}")
if (
    validation_recommended_model is not None
    and validation_recommended_model != SELECTED_MODEL_CONFIG.name
):
    print(
        f"提示：Train 内验证建议为 {validation_recommended_model}，"
        "但代码保留你的手动选择。"
    )

手动选择模型：lambdarank_32
选择备注：默认选择旧管道参考的16轮 LambdaRank，新管道结果以当前验证为准。


### 9.6 官方 Valid 安全提示

用完整 Train 分别训练手动模型与8轮基准，并在官方 Valid 上比较。下降超过0.001时只显示警告，绝不自动覆盖手动选择。

In [33]:
def ensure_numeric_validation_cache(
    start: int = VALID_START,
    stop: int = TEST_START,
    output_dir: Path = MODEL_VALID_CACHE_DIR,
    rebuild: bool = REBUILD_MODEL_VALID_CACHE,
) -> NumericTrainingCache:
    del output_dir, rebuild  # Valid 直接复用统一的 Train+Valid 缓存。
    combined_cache = ensure_numeric_training_cache()
    return numeric_cache_time_view(
        combined_cache, start, stop, role="valid"
    )


def fit_candidate_model(
    training_view: dict,
    candidate: ModelCandidate,
    lgb_module,
    seed: int = SEED,
):
    train_set = lgb_module.Dataset(
        training_view["features"],
        label=training_view["relevance"],
        group=training_view["groups"],
        weight=training_view["sample_weights"],
        feature_name=list(training_view["feature_names"]),
        free_raw_data=True,
    )
    model = lgb_module.train(
        model_candidate_parameters(
            candidate, TRAINING_SELECTION_CONFIG.label_bins, seed
        ),
        train_set,
        num_boost_round=candidate.boost_rounds,
    )
    del train_set
    gc.collect()
    return model


def evaluate_model_on_cache(
    model,
    candidate: ModelCandidate,
    cache: NumericTrainingCache,
) -> tuple[np.ndarray, dict]:
    predictions = model.predict(
        cache.features, num_iteration=candidate.boost_rounds
    ).astype(np.float32)
    rank_ic_values = per_query_rank_ic(
        predictions, cache.labels, cache.group_sizes
    )
    metrics = summarize_rank_ic(
        rank_ic_values, TRAINING_SELECTION_CONFIG.worst_block_count
    )
    return predictions, metrics


def run_official_valid_safety_check(
    train_cache: NumericTrainingCache,
    valid_cache: NumericTrainingCache,
    selected_candidate: ModelCandidate,
    training_policy: TrainingPolicy,
    warning_drop: float = OFFICIAL_VALID_WARNING_DROP,
    lgb_module=None,
):
    if lgb_module is None:
        try:
            import lightgbm as lgb_module
        except ImportError as error:
            raise ImportError("请使用 jingge_ts Anaconda 环境运行官方 Valid 检查。") from error
    training_view = build_selected_training_view(
        train_cache, VALID_START, policy=training_policy
    )
    selected_model = fit_candidate_model(
        training_view, selected_candidate, lgb_module
    )
    selected_predictions, selected_metrics = evaluate_model_on_cache(
        selected_model, selected_candidate, valid_cache
    )

    baseline_candidate = MODEL_CANDIDATE_BY_NAME[MODEL_BASELINE_NAME]
    if selected_candidate.name == MODEL_BASELINE_NAME:
        baseline_metrics = selected_metrics.copy()
    else:
        baseline_model = fit_candidate_model(
            training_view, baseline_candidate, lgb_module
        )
        baseline_predictions, baseline_metrics = evaluate_model_on_cache(
            baseline_model, baseline_candidate, valid_cache
        )
        del baseline_model, baseline_predictions
        gc.collect()

    comparison = pd.DataFrame(
        [
            {"model": MODEL_BASELINE_NAME, **baseline_metrics},
            {"model": selected_candidate.name, **selected_metrics},
        ]
    ).drop_duplicates("model", keep="last")
    rank_ic_drop = (
        baseline_metrics["mean_rank_ic"] - selected_metrics["mean_rank_ic"]
    )
    warning_triggered = bool(rank_ic_drop > warning_drop)
    return (
        selected_model, selected_predictions, selected_metrics,
        comparison, warning_triggered,
    )

In [34]:
SELECTED_VALIDATION_MODEL = None
VALID_PREDICTIONS = None
VALID_RANK_IC = None
OFFICIAL_VALID_COMPARISON = None
OFFICIAL_VALID_WARNING = False
model_valid_cache = None

if RUN_OFFICIAL_VALID_CHECK:
    if model_training_cache is None:
        model_training_cache = ensure_numeric_training_cache()
    model_valid_cache = numeric_cache_time_view(
        model_training_cache, VALID_START, TEST_START, role="valid"
    )
    (
        SELECTED_VALIDATION_MODEL,
        VALID_PREDICTIONS,
        selected_valid_metrics,
        OFFICIAL_VALID_COMPARISON,
        OFFICIAL_VALID_WARNING,
    ) = run_official_valid_safety_check(
        model_training_cache,
        model_valid_cache,
        ACTIVE_PIPELINE_CONFIG["model"],
        ACTIVE_PIPELINE_CONFIG["training"],
    )
    VALID_RANK_IC = selected_valid_metrics["mean_rank_ic"]
    display(OFFICIAL_VALID_COMPARISON)
    print(f"手动模型官方 Valid RankIC：{VALID_RANK_IC:.6f}")
    if OFFICIAL_VALID_WARNING:
        print(
            "警告：手动模型比8轮基准下降超过 "
            f"{OFFICIAL_VALID_WARNING_DROP:.3f}；不会自动替换，请自行决定。"
        )
    else:
        print("官方 Valid 未触发安全警告；仍保留手动选择。")
else:
    print("官方 Valid 安全检查未执行。")

,model,time_points,mean_rank_ic,late_rank_ic,worst_block_rank_ic
0,lambdarank_16,243,0.082150,0.075979,0.071211
1,lambdarank_32,243,0.084385,0.078038,0.074565


手动模型官方 Valid RankIC：0.084385
官方 Valid 未触发安全警告；仍保留手动选择。


## 10. 第五部分：Train+Valid 最终重训与 Test 预测

严格复用 `ACTIVE_PIPELINE_CONFIG`，不再重新选择样本、特征、时间策略或模型。最终模型只训练一次，并直接预测 Test。

### 10.1 最终训练与 Test 缓存

Train 与 Valid 标签合并为一份连续训练缓存；Test 缓存不包含标签。两者都保留全部合格股票。

In [35]:
FINAL_TRAIN_CACHE_DIR = POLICY_SCREEN_CACHE_DIR
FINAL_TEST_CACHE_DIR = NUMERIC_CACHE_DIR / "test"
RUN_FINAL_TRAINING = True
REBUILD_FINAL_TRAIN_CACHE = False
REBUILD_FINAL_TEST_CACHE = False
SAVE_FINAL_MODEL = True


@dataclass
class NumericPredictionCache:
    start: int
    stop: int
    features: np.ndarray
    time_index: np.ndarray
    stock_index: np.ndarray
    group_sizes: np.ndarray
    group_offsets: np.ndarray
    feature_names: tuple[str, ...]


def open_numeric_prediction_cache(output_dir: Path) -> NumericPredictionCache:
    output_dir = Path(output_dir)
    metadata = json.loads((output_dir / "metadata.json").read_text(encoding="utf-8"))
    if metadata["role"] != "test":
        raise ValueError(f"Test 缓存角色错误：{metadata['role']}")
    if tuple(metadata["feature_names"]) != tuple(NUMERIC_FEATURE_NAMES):
        raise ValueError("Test 缓存特征名称与当前第二部分选择不一致。")
    if metadata.get("fingerprint") != NUMERIC_FEATURE_FINGERPRINT:
        raise ValueError("Test 缓存指纹与当前补全及特征配置不一致。")

    features = np.load(output_dir / "features.npy", mmap_mode="r")
    time_index = np.load(output_dir / "time_index.npy", mmap_mode="r")
    stock_index = np.load(output_dir / "stock_index.npy", mmap_mode="r")
    group_sizes = np.load(output_dir / "group_sizes.npy").astype(np.int64)
    group_offsets = np.concatenate(
        [np.array([0], dtype=np.int64), np.cumsum(group_sizes, dtype=np.int64)]
    )
    row_count = int(group_offsets[-1])
    start = int(metadata["start"])
    stop = int(metadata["stop"])
    if group_sizes.size != stop - start:
        raise ValueError("Test 缓存分组数量与时间区间不一致。")
    if features.shape != (row_count, len(NUMERIC_FEATURE_NAMES)):
        raise ValueError(f"Test 特征形状错误：{features.shape}")
    if time_index.shape != (row_count,) or stock_index.shape != (row_count,):
        raise ValueError("Test 时间或股票索引形状错误。")
    if np.any(stock_index < 0) or np.any(stock_index >= STOCK_COUNT):
        raise ValueError("Test 股票索引越界。")
    nonempty = np.flatnonzero(group_sizes > 0)
    if nonempty.size:
        first_rows = group_offsets[nonempty]
        if not np.array_equal(time_index[first_rows], start + nonempty):
            raise ValueError("Test 时间索引与分组边界不一致。")
    return NumericPredictionCache(
        start=start, stop=stop, features=features, time_index=time_index,
        stock_index=stock_index, group_sizes=group_sizes,
        group_offsets=group_offsets, feature_names=tuple(metadata["feature_names"]),
    )


def ensure_final_training_cache(
    start: int = TRAIN_START,
    stop: int = TEST_START,
    output_dir: Path = FINAL_TRAIN_CACHE_DIR,
    rebuild: bool = REBUILD_FINAL_TRAIN_CACHE,
) -> NumericTrainingCache:
    output_dir = Path(output_dir)
    required = [
        output_dir / name
        for name in (
            "features.npy", "labels.npy", "time_index.npy",
            "query_weights.npy", "group_sizes.npy", "metadata.json",
        )
    ]
    complete = all(path.exists() for path in required)
    if complete and not rebuild:
        cache = open_numeric_training_cache(output_dir, expected_role="train")
        if cache.start != start or cache.stop != stop:
            raise ValueError("Train+Valid 缓存范围不匹配；请显式启用 rebuild。")
        return cache
    if not complete and any(path.exists() for path in required) and not rebuild:
        raise FileExistsError("发现不完整 Train+Valid 缓存；确认后启用 rebuild。")
    materialize_numeric_feature_cache(
        start, stop, "train", output_dir, stock_cap=None, overwrite=rebuild
    )
    return open_numeric_training_cache(output_dir, expected_role="train")


def ensure_final_test_cache(
    start: int = TEST_START,
    stop: int = T,
    output_dir: Path = FINAL_TEST_CACHE_DIR,
    rebuild: bool = REBUILD_FINAL_TEST_CACHE,
) -> NumericPredictionCache:
    output_dir = Path(output_dir)
    required = [
        output_dir / name
        for name in (
            "features.npy", "time_index.npy", "stock_index.npy",
            "group_sizes.npy", "metadata.json",
        )
    ]
    complete = all(path.exists() for path in required)
    if complete and not rebuild:
        cache = open_numeric_prediction_cache(output_dir)
        if cache.start != start or cache.stop != stop:
            raise ValueError("Test 缓存范围不匹配；请显式启用 rebuild。")
        return cache
    if not complete and any(path.exists() for path in required) and not rebuild:
        raise FileExistsError("发现不完整 Test 缓存；确认后启用 rebuild。")
    materialize_numeric_feature_cache(
        start, stop, "test", output_dir, stock_cap=None, overwrite=rebuild
    )
    return open_numeric_prediction_cache(output_dir)

### 10.2 最终模型训练函数

模型文本通过内存字符串原子写入，避免 LightGBM 在 Windows 中文路径下直接保存时的兼容问题。

In [36]:
def save_model_text_atomic(
    model,
    candidate: ModelCandidate,
    path: Path = MODEL_PATH,
) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model_text = model.model_to_string(num_iteration=candidate.boost_rounds)
    if not model_text.strip():
        raise ValueError("模型文本为空。")
    partial_path = path.with_suffix(path.suffix + ".partial")
    partial_path.write_text(model_text, encoding="utf-8")
    partial_path.replace(path)
    if path.stat().st_size <= 0:
        raise IOError("模型文件写入失败。")
    return path


def run_final_refit_and_predict(
    train_cache: NumericTrainingCache,
    test_cache: NumericPredictionCache,
    training_policy: TrainingPolicy,
    model_candidate: ModelCandidate,
    lgb_module=None,
):
    if lgb_module is None:
        try:
            import lightgbm as lgb_module
        except ImportError as error:
            raise ImportError("请使用 jingge_ts Anaconda 环境执行最终训练。") from error
    if train_cache.stop != TEST_START:
        raise ValueError("最终训练缓存必须结束于 TEST_START。")
    if test_cache.start != TEST_START or test_cache.stop != T:
        raise ValueError("最终 Test 缓存范围错误。")

    training_view = build_selected_training_view(
        train_cache, TEST_START, policy=training_policy
    )
    final_model = fit_candidate_model(
        training_view, model_candidate, lgb_module, seed=SEED
    )
    raw_test_predictions = final_model.predict(
        test_cache.features, num_iteration=model_candidate.boost_rounds
    ).astype(np.float32)
    if raw_test_predictions.shape != (int(test_cache.group_sizes.sum()),):
        raise ValueError("Test 原始预测行数与缓存不一致。")
    if not np.all(np.isfinite(raw_test_predictions)):
        raise ValueError("Test 原始预测包含非有限值。")
    del training_view
    gc.collect()
    return final_model, raw_test_predictions

## 11. 第六部分：提交矩阵生成与验收

每个 Test 时间截面的原始模型分数独立转换为 `[0, 1]` 百分位排名；官方评价池外严格填充 `0.5`。

In [37]:
def build_submission_matrix(
    raw_predictions: np.ndarray,
    test_cache: NumericPredictionCache,
    evaluation_mask: np.ndarray,
    stock_count: int,
) -> np.ndarray:
    raw_predictions = np.asarray(raw_predictions, dtype=np.float64)
    evaluation_mask = np.asarray(evaluation_mask, dtype=bool)
    time_count = test_cache.stop - test_cache.start
    if evaluation_mask.shape != (time_count, stock_count):
        raise ValueError("Test 评价掩码形状错误。")
    if raw_predictions.shape != (int(test_cache.group_sizes.sum()),):
        raise ValueError("原始预测长度与 Test 缓存不一致。")
    if not np.all(np.isfinite(raw_predictions)):
        raise ValueError("原始预测包含非有限值。")

    output = np.full((time_count, stock_count), 0.5, dtype=np.float32)
    for local_time, group_size in enumerate(test_cache.group_sizes):
        left = int(test_cache.group_offsets[local_time])
        right = int(test_cache.group_offsets[local_time + 1])
        group_size = int(group_size)
        stocks = np.asarray(test_cache.stock_index[left:right], dtype=np.int32)
        expected_stocks = np.flatnonzero(evaluation_mask[local_time]).astype(np.int32)
        if group_size != stocks.size or not np.array_equal(stocks, expected_stocks):
            raise ValueError(
                f"时间 {test_cache.start + local_time} 的评价股票与缓存不一致。"
            )
        if group_size == 0:
            continue
        if group_size == 1:
            percentiles = np.array([0.5], dtype=np.float32)
        else:
            ranks = rankdata(raw_predictions[left:right], method="average")
            percentiles = (
                (ranks - 1.0) / (group_size - 1.0)
            ).astype(np.float32)
        output[local_time, stocks] = percentiles
    return output


def validate_submission_matrix(
    prediction: np.ndarray,
    evaluation_mask: np.ndarray,
) -> pd.Series:
    prediction = np.asarray(prediction)
    evaluation_mask = np.asarray(evaluation_mask, dtype=bool)
    expected_shape = (T - TEST_START, STOCK_COUNT)
    if prediction.shape != expected_shape:
        raise ValueError(f"提交形状错误：{prediction.shape} != {expected_shape}")
    if prediction.dtype != np.float32:
        raise TypeError(f"提交类型必须是 float32，实际为 {prediction.dtype}")
    if evaluation_mask.shape != expected_shape:
        raise ValueError("评价掩码与提交形状不一致。")
    if not np.all(np.isfinite(prediction)):
        raise ValueError("提交包含 NaN 或无穷值。")
    if not np.all(prediction[~evaluation_mask] == np.float32(0.5)):
        raise ValueError("非评价位置没有全部填充 0.5。")
    evaluated = prediction[evaluation_mask]
    if evaluated.size == 0:
        raise ValueError("Test 没有评价位置。")
    if np.min(evaluated) < 0.0 or np.max(evaluated) > 1.0:
        raise ValueError("评价位置预测超出 [0, 1]。")
    return pd.Series(
        {
            "shape": str(prediction.shape),
            "dtype": str(prediction.dtype),
            "evaluation_positions": int(np.count_nonzero(evaluation_mask)),
            "neutral_non_evaluation_positions": int(
                np.count_nonzero(prediction[~evaluation_mask] == 0.5)
            ),
            "non_evaluation_all_neutral_0_5": True,
            "minimum": float(prediction.min()),
            "maximum": float(prediction.max()),
            "mean": float(prediction.mean()),
            "all_finite": True,
        },
        name="submission_validation",
    )

### 11.1 最终流程自检

合成测试检查逐时点排名、股票索引映射、非评价位置0.5和提交矩阵类型。

In [38]:
def run_final_pipeline_self_test() -> None:
    toy_groups = np.array([3, 2], dtype=np.int64)
    toy_offsets = np.concatenate([[0], np.cumsum(toy_groups)])
    toy_stocks = np.array([0, 2, 3, 1, 3], dtype=np.int32)
    toy_cache = NumericPredictionCache(
        start=10, stop=12,
        features=np.zeros((5, 2), dtype=np.float32),
        time_index=np.array([10, 10, 10, 11, 11], dtype=np.int32),
        stock_index=toy_stocks,
        group_sizes=toy_groups, group_offsets=toy_offsets,
        feature_names=("f0", "f1"),
    )
    toy_mask = np.array(
        [[True, False, True, True], [False, True, False, True]], dtype=bool
    )
    toy_raw = np.array([0.2, 0.8, 0.5, -1.0, 1.0], dtype=np.float32)
    toy_submission = build_submission_matrix(toy_raw, toy_cache, toy_mask, 4)
    assert toy_submission.shape == (2, 4)
    assert toy_submission.dtype == np.float32
    assert np.all(toy_submission[~toy_mask] == 0.5)
    assert np.allclose(toy_submission[0, [0, 2, 3]], [0.0, 1.0, 0.5])
    assert np.allclose(toy_submission[1, [1, 3]], [0.0, 1.0])
    print("最终训练与提交矩阵接口自检通过。")


run_final_pipeline_self_test()

最终训练与提交矩阵接口自检通过。


### 11.2 执行最终训练并保存提交

只保存当前实验的 `model.txt` 与 `prediction.npy`。正式提交目录不会被自动覆盖。

In [39]:
FINAL_MODEL = None
RAW_TEST_PREDICTIONS = None
TEST_PREDICTIONS = None
SUBMISSION_VALIDATION = None
final_training_cache = None
final_test_cache = None

if RUN_FINAL_TRAINING:
    final_training_cache = model_training_cache or training_policy_cache
    if final_training_cache is None:
        final_training_cache = ensure_final_training_cache()
    final_test_cache = ensure_final_test_cache()
    FINAL_MODEL, RAW_TEST_PREDICTIONS = run_final_refit_and_predict(
        final_training_cache,
        final_test_cache,
        ACTIVE_PIPELINE_CONFIG["training"],
        ACTIVE_PIPELINE_CONFIG["model"],
    )
    if SAVE_FINAL_MODEL:
        saved_model_path = save_model_text_atomic(
            FINAL_MODEL, SELECTED_MODEL_CONFIG, MODEL_PATH
        )
        print(f"最终模型已保存：{saved_model_path.resolve()}")

    official_evaluation_mask = np.asarray(
        mask_y[TEST_START:T], dtype=bool
    )
    if not np.any(official_evaluation_mask):
        raise ValueError("Test 的 mask_y 没有任何评价位置。")
    TEST_PREDICTIONS = build_submission_matrix(
        RAW_TEST_PREDICTIONS,
        final_test_cache,
        official_evaluation_mask,
        STOCK_COUNT,
    )
    SUBMISSION_VALIDATION = validate_submission_matrix(
        TEST_PREDICTIONS, official_evaluation_mask
    )
    saved_prediction_path = save_prediction(TEST_PREDICTIONS, PREDICTION_PATH)
    reloaded_prediction = np.load(saved_prediction_path, allow_pickle=False)
    reloaded_validation = validate_submission_matrix(
        reloaded_prediction, official_evaluation_mask
    )
    if not np.array_equal(reloaded_prediction, TEST_PREDICTIONS):
        raise IOError("提交文件回读后与内存预测不一致。")
    SUBMISSION_VALIDATION["path"] = str(saved_prediction_path.resolve())
    SUBMISSION_VALIDATION["file_size_mb"] = (
        saved_prediction_path.stat().st_size / 1024**2
    )
    SUBMISSION_VALIDATION["np_load_verified"] = bool(
        reloaded_validation["all_finite"]
    )
    ACTIVE_PIPELINE_CONFIG["artifacts"] = {
        "model_path": str(MODEL_PATH.resolve()),
        "prediction_path": str(PREDICTION_PATH.resolve()),
    }
    display(SUBMISSION_VALIDATION)
    print("最终训练与提交文件验收完成。")
else:
    print("最终训练未执行。")

最终模型已保存：D:\google_dl\book\友安杯\04_results\exp_008_new_method\model.txt
预测已保存：D:\google_dl\book\友安杯\04_results\exp_008_new_method\prediction.npy


shape                                                                     (442, 5282)
dtype                                                                         float32
evaluation_positions                                                          2042538
neutral_non_evaluation_positions                                               292106
non_evaluation_all_neutral_0_5                                                   True
minimum                                                                           0.0
maximum                                                                           1.0
mean                                                                              0.5
all_finite                                                                       True
path                                D:\google_dl\book\友安杯\04_results\exp_008_new_m...
file_size_mb                                                                 8.906082
np_load_verified                                      

最终训练与提交文件验收完成。


## 完成状态

运行至此后，`ACTIVE_PIPELINE_CONFIG`、`FINAL_MODEL`、`VALID_PREDICTIONS`、`TEST_PREDICTIONS` 与当前实验的 `prediction.npy` 均可直接使用。Notebook 不会自动覆盖 `04_results/final_submission/prediction.npy`。